## Raw EEG Preprocessing (ICA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import mat73
import scipy.io
from scipy.signal import butter, filtfilt, iirnotch, welch
from sklearn.decomposition import FastICA

def extract_eeg_info(eeg_struct):
    """Extract EEG data fields into a dictionary."""
    return {
        'subject': eeg_struct.get('subject', 'Unknown'),
        'group': eeg_struct.get('group', 'Unknown'),
        'condition': eeg_struct.get('condition', 'Unknown'),
        'nbchan': int(eeg_struct.get('nbchan', 0)),
        'trials': int(eeg_struct.get('trials', 0)),
        'pnts': int(eeg_struct.get('pnts', 0)),
        'srate': float(eeg_struct.get('srate', 0)),
        'data': eeg_struct.get('data'),
        'data_shape': eeg_struct['data'].shape if 'data' in eeg_struct else (0,0),
        'times': eeg_struct.get('times')
    }

def analyze_eeg_data(file_path):
    try:
        try:
            data = mat73.loadmat(file_path)
        except:
            data = scipy.io.loadmat(file_path)
            
        if 'EEG' in data:
            eeg_content = data['EEG']
            if hasattr(eeg_content, 'dtype') and eeg_content.shape == (1, 1):
                eeg_content = eeg_content[0, 0]
                eeg_content = {name: eeg_content[name] for name in eeg_content.dtype.names}
            return extract_eeg_info(eeg_content)
    except Exception as e:
        print(f"Failed to load {os.path.basename(file_path)}: {e}")
    return None

def plot_eeg_signal(eeg_info, subject_name, num_channels=20):
    """Plot channels with auto-scaling to prevent flat-line effect."""
    data = eeg_info['data']
    if data is None: return

    if len(data.shape) == 3: data = data[0, :, :]
        
    # Downsample for speed
    step = 30
    plot_data = data[:num_channels, ::step]
    
    plt.figure(figsize=(12, 6))
    
    # Auto-scale: Normalize each channel so waves are visible
    for i in range(min(num_channels, plot_data.shape[0])):
        ch_data = plot_data[i]
        # Normalize: (data - mean) / std dev makes waves visible
        std_val = np.nanstd(ch_data)
        normalized = (ch_data - np.nanmean(ch_data)) / (std_val if std_val > 0 else 1)
        plt.plot(normalized + (i * 2.5), color='black', linewidth=0.5)
        
    plt.title(f"EEG Wave Check (Normalized): {subject_name}")
    plt.xlabel("Time (Downsampled)")
    plt.yticks([])
    plt.grid(True, axis='x', alpha=0.3)
    plt.show()

def main():
    current_dir = os.path.join(os.getcwd(), 'PD_REST')
    mat_files = [f for f in os.listdir(current_dir) if f.endswith('.mat') and not f.endswith('1.mat')]
    
    # Pick 4 random subjects
    selected_files = ['803_2_PD_REST.mat', '818_2_PD_REST.mat']
    
    plot_data = None
    plot_subject = None

    for file in selected_files:
        eeg_info = analyze_eeg_data(os.path.join(current_dir, file))
        
        # FIX: Check for NoneType before accessing
        if eeg_info and eeg_info['data'] is not None and eeg_info['data'].size > 0:
            print(f"Successfully loaded {file} | Shape: {eeg_info['data_shape']}")
            if plot_data is None:
                plot_data, plot_subject = eeg_info, file
        else:
            print(f"Skipping {file}: No valid data found.")

    if plot_data:
        plot_eeg_signal(plot_data, plot_subject)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import mat73
import scipy.io
from mne.preprocessing import ICA
import mne
from scipy.signal import butter, filtfilt, iirnotch, welch


# --------------------------------------------------
# EXTRACT CHANNEL LABELS
# --------------------------------------------------

def extract_channel_labels(chanlocs):
    labels = []
    for ch in chanlocs[0]:
        try:
            label = ch[0]
            while isinstance(label, np.ndarray):
                if label.size == 0:
                    label = "Unknown"
                    break
                label = label.flat[0]
            labels.append(str(label))
        except Exception:
            labels.append("Unknown")
    return labels


# --------------------------------------------------
# TOPOMAP PLOT
# --------------------------------------------------

def plot_ica_topomaps_from_processor(proc, n_components=10):
    if not hasattr(proc, "ica") or not hasattr(proc, "raw"):
        raise ValueError("ICA not fitted yet — call fit_ica() first.")

    ica  = proc.ica
    raw  = proc.raw

    components        = ica.get_components()
    montage           = mne.channels.make_standard_montage("standard_1020")
    montage_positions = montage.get_positions()["ch_pos"]

    valid_channels = [ch for ch in raw.ch_names if ch in montage_positions]
    if not valid_channels:
        raise RuntimeError(
            f"No channels matched standard_1020.\n"
            f"Channels in raw: {raw.ch_names[:10]}"
        )

    pos   = np.array([montage_positions[ch][:2] for ch in valid_channels])
    picks = [raw.ch_names.index(ch) for ch in valid_channels]

    n_plot = min(n_components, components.shape[1])
    fig, axes = plt.subplots(1, n_plot, figsize=(3 * n_plot, 3))
    axes = np.atleast_1d(axes)

    for i in range(n_plot):
        mne.viz.plot_topomap(
            components[picks, i], pos,
            axes=axes[i], show=False, extrapolate='head'
        )
        title = f"IC {i}" + (" ✗" if i in ica.exclude else "")
        axes[i].set_title(title)

    plt.suptitle("ICA Component Topomaps", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# --------------------------------------------------
# MAIN CLASS
# --------------------------------------------------

class PDBrainProcessor:

    NON_EEG = {'VEOG', 'HEOG', 'X', 'Y', 'Z', 'EMG', 'ECG', 'EOG', 'Status', 'Unknown'}

    def __init__(self, srate):
        self.srate = float(np.squeeze(srate))
        self.bands = {
            'Delta':        (0.5, 4),
            'Theta':        (4,   8),
            'Alpha':        (8,  13),
            'Beta (Motor)': (13, 30),
            'Tremor':       (3,   6),
        }
        self.raw          = None
        self.ica          = None
        self.clean_labels = None
        self.cleaned_data = None

    # --------------------------------------------------
    # INTERNAL HELPERS
    # --------------------------------------------------

    def _prepare_data(self, raw_data, labels):
        data = np.squeeze(raw_data)
        data = data[:, :50000]
        data = self._ensure_channel_first(data)

        n_ch = data.shape[0]
        if len(labels) >= n_ch:
            labels = labels[:n_ch]
        else:
            labels = labels + [f"EEG{i}" for i in range(len(labels), n_ch)]

        keep   = [i for i, l in enumerate(labels) if l not in self.NON_EEG]
        data   = data[keep, :]
        labels = [labels[i] for i in keep]

        print(f"  Channels kept : {len(labels)}")
        print(f"  Data range    : {data.min():.1f} – {data.max():.1f}")
        return data, labels

    def _ensure_channel_first(self, data):
        if data.shape[0] > data.shape[1]:
            data = data.T
        return data
    
    def auto_detect_bad_components(self):
        if self.ica is None or self.raw is None:
            raise RuntimeError("Call fit_ica() first.")

        bad_components = set()

        # Detect eye blinks / EOG
        try:
            eog_indices, _ = self.ica.find_bads_eog(
                self.raw,
                ch_name   = self._pick_frontal_channel(),
                threshold = 3.0,
                verbose   = False
            )
            bad_components.update(eog_indices)
        except Exception:
            pass

        # Detect muscle artifacts
        try:
            muscle_indices, _ = self.ica.find_bads_muscle(
                self.raw,
                threshold = 0.5,
                verbose   = False
            )
            bad_components.update(muscle_indices)
        except Exception:
            pass

        return sorted(bad_components)


    def _pick_frontal_channel(self):
        for ch in ['Fp1', 'Fp2', 'AF3', 'AF4']:
            if ch in self.raw.ch_names:
                return ch
        return self.raw.ch_names[0]
    
    def get_component_summary(self, bad_components):
        """
        Summarize how many / which ICA components were removed
        for this subject. Used for the cross-subject analysis
        requested by the mentor.
        """
        return {
            'n_total_components': self.ica.n_components_,
            'n_removed'          : len(bad_components),
            'removed_indices'    : list(bad_components),
            'fraction_removed'   : len(bad_components) / self.ica.n_components_,
        }

    # --------------------------------------------------
    # FILTERING
    # --------------------------------------------------

    def apply_bandpass(self, data, low=1.0, high=40.0):
        nyq  = 0.5 * self.srate
        b, a = butter(4, [low / nyq, high / nyq], btype='band')
        return filtfilt(b, a, data, axis=-1)

    def apply_notch(self, data, freq=50.0):
        nyq  = 0.5 * self.srate
        b, a = iirnotch(freq / nyq, 35)
        return filtfilt(b, a, data, axis=-1)

    def apply_avg_ref(self, data):
        return data - np.mean(data, axis=0, keepdims=True)

    # --------------------------------------------------
    # STEP 1 — fit ICA
    # --------------------------------------------------

    def fit_ica(self, raw_data, labels):
        print("\n[1/2] Preparing and filtering data...")
        data, labels = self._prepare_data(raw_data, labels)
        self.clean_labels = labels

        data = self.apply_bandpass(data)
        data = self.apply_notch(data)
        data = self.apply_avg_ref(data)
        self._filtered_data = data

        print("\n[2/2] Fitting ICA...")
        info = mne.create_info(ch_names=labels, sfreq=self.srate, ch_types='eeg')
        raw  = mne.io.RawArray(data, info, verbose=False)
        raw.set_montage(
            mne.channels.make_standard_montage("standard_1020"),
            on_missing='ignore'
        )
        self.raw = raw

        ica = ICA(
            n_components=0.999999,
            method='infomax',
            fit_params=dict(extended=True),
            random_state=42,
            max_iter='auto'
        )
        ica.fit(raw, decim=4)
        self.ica = ica

        sources = ica.get_sources(raw).get_data().T
        self._sources = sources

        print(f"  ICA fitted   : {ica.n_components_} components")
        print(f"  Sources shape: {sources.shape}")
        print("\nInspect with plot_ica_timeseries() and plot_ica_topomaps_from_processor()")
        print("Then call remove_artifacts(bad=[0, 1, 2])")

        return sources

    # --------------------------------------------------
    # STEP 2 — remove bad ICA components
    # --------------------------------------------------

    def remove_artifacts(self, bad_components):
        if self.ica is None or self.raw is None:
            raise RuntimeError("Call fit_ica() before remove_artifacts().")

        print(f"\nRemoving ICA components: {bad_components}")

        raw_clean = self.raw.copy()
        self.ica.exclude = bad_components
        self.ica.apply(raw_clean)

        # Update self.raw so interpolation works on the ICA-cleaned signal
        self.raw          = raw_clean
        self.cleaned_data = raw_clean.get_data()

        print(f"Cleaned data shape: {self.cleaned_data.shape}")
        return self.cleaned_data

    # --------------------------------------------------
    # STEP 3 — interpolate bad channels
    # --------------------------------------------------

    def interpolate_bad_channels(self, z_threshold=3.0, max_bad_fraction=0.1):
        """
        max_bad_fraction : float
            Maximum fraction of channels allowed to be interpolated.
            If more are detected, skip interpolation for this subject.
            Default 0.1 = max 10% of channels.
        """
        if self.raw is None:
            raise RuntimeError("Call fit_ica() and remove_artifacts() first.")

        data   = self.raw.get_data()
        labels = self.raw.ch_names
        n_ch   = data.shape[0]

        # ── Criterion 1: high variance ────────────────────────────────
        variances = np.var(data, axis=1)
        mean_var  = np.mean(variances)
        std_var   = np.std(variances)
        bad_var   = [labels[i] for i, v in enumerate(variances)
                    if v > mean_var + z_threshold * std_var]

        # ── Criterion 2: spikes — use stricter threshold ──────────────
        ch_means  = np.mean(data, axis=1, keepdims=True)
        ch_stds   = np.std(data,  axis=1, keepdims=True)
        z_scores  = np.abs((data - ch_means) / (ch_stds + 1e-8))
        bad_spike = [labels[i] for i in range(n_ch)
                    if np.max(z_scores[i]) > 20]   # raised from 10 → 20

        bad_channels = list(set(bad_var + bad_spike))

        # ── Safety check: skip if too many channels flagged ───────────
        max_bad = int(n_ch * max_bad_fraction)
        if len(bad_channels) > max_bad:
            print(f"  ⚠ Too many bad channels ({len(bad_channels)}/{n_ch}) "
                f"— skipping interpolation to protect data integrity.")
            return self.cleaned_data

        if not bad_channels:
            return self.cleaned_data

        print(f"  Interpolating {len(bad_channels)} channels: {bad_channels}")

        raw_interp = self.raw.copy()
        raw_interp.info['bads'] = bad_channels
        raw_interp.interpolate_bads(reset_bads=True, verbose=False)

        self.raw          = raw_interp
        self.cleaned_data = raw_interp.get_data()
        return self.cleaned_data

    # --------------------------------------------------
    # INSPECTION PLOTS  (between fit_ica and remove_artifacts)
    # --------------------------------------------------

    def plot_ica_timeseries(self, num_components=10):
        if not hasattr(self, '_sources'):
            raise RuntimeError("Call fit_ica() first.")
        sources = self._sources
        n = min(num_components, sources.shape[1])
        fig, axes = plt.subplots(n, 1, figsize=(14, 2 * n), sharex=True)
        axes = np.atleast_1d(axes)
        for i in range(n):
            axes[i].plot(sources[:3000, i], lw=0.7, color='steelblue')
            axes[i].set_ylabel(f"IC {i}", fontsize=8, rotation=0, labelpad=28)
            axes[i].set_yticks([])
            for sp in ['top', 'right', 'left']:
                axes[i].spines[sp].set_visible(False)
        axes[-1].set_xlabel("Samples")
        plt.suptitle("ICA Component Time Series — inspect before removing", y=1.01)
        plt.tight_layout()
        plt.show()

    # --------------------------------------------------
    # RESULT PLOTS  (after remove_artifacts / interpolate)
    # --------------------------------------------------

    def plot_eeg_cleaned(self, subject_name, n_channels=20):
        if self.cleaned_data is None:
            raise RuntimeError("Call remove_artifacts() first.")
        step      = 20
        plot_data = self.cleaned_data[:n_channels, ::step]
        spacing   = np.std(self.cleaned_data) * 5

        plt.figure(figsize=(14, 7))
        for i in range(plot_data.shape[0]):
            plt.plot(plot_data[i] + i * spacing, lw=0.6)
        plt.title(f"Cleaned EEG — {subject_name}")
        plt.xlabel("Samples (decimated ×20)")
        plt.yticks(
            [i * spacing for i in range(plot_data.shape[0])],
            self.clean_labels[:n_channels],
            fontsize=7
        )
        plt.tight_layout()
        plt.show()

    def plot_psd(self, channel=0):
        if self.cleaned_data is None:
            raise RuntimeError("Call remove_artifacts() first.")
        sig     = self.cleaned_data[channel]
        f, pxx  = welch(sig, fs=self.srate, nperseg=2048)
        ch_name = self.clean_labels[channel] if self.clean_labels else str(channel)
        plt.figure(figsize=(8, 4))
        plt.semilogy(f, pxx, lw=1.2)
        plt.title(f"PSD — {ch_name}")
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("Power spectral density")
        plt.xlim(0, 50)
        plt.tight_layout()
        plt.show()

    def plot_before_after(self, channel=0):
        if self.cleaned_data is None or not hasattr(self, '_filtered_data'):
            raise RuntimeError("Call fit_ica() and remove_artifacts() first.")
        ch_name = self.clean_labels[channel] if self.clean_labels else str(channel)
        f_before, p_before = welch(self._filtered_data[channel], fs=self.srate, nperseg=2048)
        f_after,  p_after  = welch(self.cleaned_data[channel],   fs=self.srate, nperseg=2048)
        plt.figure(figsize=(9, 4))
        plt.semilogy(f_before, p_before, lw=1.0, label='Before ICA removal', alpha=0.6)
        plt.semilogy(f_after,  p_after,  lw=1.2, label='After ICA removal + interpolation')
        plt.title(f"PSD before vs after — {ch_name}")
        plt.xlabel("Frequency (Hz)")
        plt.ylabel("Power spectral density")
        plt.xlim(0, 50)
        plt.legend()
        plt.tight_layout()
        plt.show()


# --------------------------------------------------
# LOAD EEG
# --------------------------------------------------

def load_eeg(path):
    try:
        data = mat73.loadmat(path)
    except Exception:
        data = scipy.io.loadmat(path)
    eeg = data.get('EEG', {})
    if hasattr(eeg, 'dtype'):
        eeg = {n: eeg[0, 0][n] for n in eeg.dtype.names}
    return eeg


# --------------------------------------------------
# MAIN
# --------------------------------------------------

def main():
    path           = os.path.join(os.getcwd(), 'PD_REST')
    selected_files = ['803_2_PD_REST.mat']
    bad_components = [0, 1, 2]   # IC0=eye blink, IC1=eye movement, IC2=slow drift

    for file in selected_files:
        info = load_eeg(os.path.join(path, file))

        if info.get('data') is None or info.get('srate') is None:
            print(f"Missing data or srate: {file}")
            continue

        print(f"\n{'='*50}\nProcessing: {file}\n{'='*50}")

        labels = extract_channel_labels(info['chanlocs'])
        labels = [l for l in labels if l != 'Unknown']
        print(f"Labels ({len(labels)}): {labels[:6]} ...")

        proc = PDBrainProcessor(info['srate'])

        # STEP 1 — filter + fit ICA once
        proc.fit_ica(info['data'], labels)


        # STEP 2 — inspect (look at these, then decide bad_components)
        proc.plot_ica_timeseries(num_components=10)
        plot_ica_topomaps_from_processor(proc, n_components=10)

        # STEP 3 — remove bad ICA components
        proc.remove_artifacts(bad_components)

        # STEP 4 — auto-detect and interpolate bad channels
        proc.interpolate_bad_channels(z_threshold=3.0)

        # STEP 5 — visualise final result
        proc.plot_eeg_cleaned(subject_name=file, n_channels=20)
        proc.plot_psd(channel=0)
        proc.plot_before_after(channel=0)


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import os


# --------------------------------------------------
# SAVE CLEANED DATA
# --------------------------------------------------

def save_cleaned_data(proc, subject_id, output_dir='cleaned'):
    """
    Save cleaned EEG data, channel labels, and sampling rate to a .npz file.

    Parameters
    ----------
    proc        : PDBrainProcessor  — must have cleaned_data set
    subject_id  : str               — e.g. '803_2_PD_REST'
    output_dir  : str               — folder to save into
    """
    if proc.cleaned_data is None:
        raise RuntimeError("No cleaned data — run the full pipeline first.")

    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{subject_id}.npz")

    np.savez(
        save_path,
        data    = proc.cleaned_data,          # (n_channels, n_samples)
        labels  = np.array(proc.clean_labels),
        srate   = np.array(proc.srate),
    )
    print(f"Saved cleaned data → {save_path}")
    print(f"  Shape  : {proc.cleaned_data.shape}")
    print(f"  Srate  : {proc.srate} Hz")
    print(f"  Labels : {proc.clean_labels[:5]} ...")
    return save_path


# --------------------------------------------------
# EPOCH DATA
# --------------------------------------------------

def epoch_data(proc, epoch_sec=2.0, overlap=0.5):
    """
    Cut cleaned EEG into fixed-length overlapping epochs.

    Parameters
    ----------
    proc       : PDBrainProcessor   — must have cleaned_data and srate set
    epoch_sec  : float              — epoch length in seconds (default 2.0)
    overlap    : float              — overlap fraction 0–1 (default 0.5 = 50%)

    Returns
    -------
    epochs : np.ndarray  shape (n_epochs, n_channels, n_times)
    """
    if proc.cleaned_data is None:
        raise RuntimeError("No cleaned data — run the full pipeline first.")

    data       = proc.cleaned_data                        # (n_channels, n_samples)
    srate      = proc.srate
    epoch_len  = int(epoch_sec * srate)                   # samples per epoch
    step       = int(epoch_len * (1 - overlap))           # step size in samples
    n_channels, n_samples = data.shape

    # Slide a window across the recording
    starts = np.arange(0, n_samples - epoch_len + 1, step)
    epochs = np.array([data[:, s:s + epoch_len] for s in starts])

    print(f"\nEpoching complete:")
    print(f"  Epoch length : {epoch_sec}s  ({epoch_len} samples)")
    print(f"  Overlap      : {int(overlap*100)}%  (step = {step} samples)")
    print(f"  Total epochs : {len(epochs)}")
    print(f"  Epoch shape  : {epochs.shape}  (n_epochs, n_channels, n_times)")

    return epochs


# --------------------------------------------------
# EXTRACT BAND POWER FEATURES
# --------------------------------------------------

def extract_band_power(epochs, srate, bands):
    """
    Compute mean band power per epoch per channel using Welch PSD.

    Parameters
    ----------
    epochs : np.ndarray   shape (n_epochs, n_channels, n_times)
    srate  : float        sampling rate in Hz
    bands  : dict         e.g. {'Alpha': (8, 13), 'Beta': (13, 30)}

    Returns
    -------
    features : np.ndarray  shape (n_epochs, n_channels * n_bands)
    feat_names : list      column names e.g. ['Fp1_Delta', 'Fp1_Theta', ...]
    """
    n_epochs, n_channels, n_times = epochs.shape
    band_names = list(bands.keys())
    n_bands    = len(band_names)

    features = np.zeros((n_epochs, n_channels * n_bands))

    for e in range(n_epochs):
        for ch in range(n_channels):
            sig        = epochs[e, ch, :]
            f, pxx     = welch(sig, fs=srate, nperseg=min(256, n_times))

            for b, band_name in enumerate(band_names):
                low, high  = bands[band_name]
                idx        = np.where((f >= low) & (f <= high))[0]
                band_power = np.mean(pxx[idx]) if len(idx) > 0 else 0.0
                col        = ch * n_bands + b
                features[e, col] = band_power

    print(f"\nBand power extraction complete:")
    print(f"  Features shape : {features.shape}  (n_epochs, n_channels × n_bands)")
    print(f"  Bands          : {band_names}")

    return features


def make_feature_names(clean_labels, bands):
    """Generate column names like Fp1_Alpha, Fz_Beta, ..."""
    return [
        f"{ch}_{band}"
        for ch in clean_labels
        for band in bands.keys()
    ]


# --------------------------------------------------
# SAVE EPOCHS + FEATURES
# --------------------------------------------------

def save_epochs_and_features(epochs, features, feat_names,
                              subject_id, label, output_dir='epochs'):
    """
    Save epochs and band power features for one subject.

    Parameters
    ----------
    epochs      : np.ndarray  (n_epochs, n_channels, n_times)
    features    : np.ndarray  (n_epochs, n_features)
    feat_names  : list of str
    subject_id  : str         e.g. '803_2'
    label       : int         1 = PD patient, 0 = healthy control
    output_dir  : strA
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{subject_id}.npz")

    np.savez(
        save_path,
        epochs     = epochs,
        features   = features,
        feat_names = np.array(feat_names),
        label      = np.array(label),
    )
    print(f"Saved epochs + features → {save_path}")
    print(f"  Epochs   : {epochs.shape}")
    print(f"  Features : {features.shape}")
    return save_path


# --------------------------------------------------
# VISUALISE EPOCHS
# --------------------------------------------------

def plot_epoch_band_power(features, feat_names, epoch_idx=0):
    """
    Bar chart of band power for one epoch across all channels and bands.
    Useful sanity check after feature extraction.
    """
    plt.figure(figsize=(14, 4))
    plt.bar(range(len(feat_names)), features[epoch_idx], width=1.0)
    plt.title(f"Band power — epoch {epoch_idx}")
    plt.xlabel("Channel × Band")
    plt.ylabel("Power")

    # Only label every N-th tick to keep it readable
    n      = len(feat_names)
    step   = max(1, n // 20)
    ticks  = list(range(0, n, step))
    plt.xticks(ticks, [feat_names[i] for i in ticks], rotation=45, fontsize=7)

    plt.tight_layout()
    plt.show()

In [ ]:
def main():

    path = os.path.join(os.getcwd(), "PD_REST")

    # Process every subject
    selected_files = sorted(
        f for f in os.listdir(path)
        if f.endswith(".mat")
    )

    component_summary = []

    for file in selected_files:

        info = load_eeg(os.path.join(path, file))

        if info.get("data") is None or info.get("srate") is None:
            print(f"Skipping {file} (missing data)")
            continue

        print("\n" + "=" * 60)
        print(f"Processing: {file}")
        print("=" * 60)

        labels = extract_channel_labels(info["chanlocs"])
        labels = [l for l in labels if l != "Unknown"]

        proc = PDBrainProcessor(info["srate"])

        # -------------------------------------------------
        # STEP 1 — Fit ICA
        # -------------------------------------------------
        proc.fit_ica(info["data"], labels)

        # -------------------------------------------------
        # STEP 2 — Automatically detect bad ICA components
        # -------------------------------------------------
        bad_components = proc.auto_detect_bad_components()

        print(f"Detected ICA components: {bad_components}")

        # Save statistics
        summary = proc.get_component_summary(bad_components)
        summary["subject"] = file
        component_summary.append(summary)

        # -------------------------------------------------
        # OPTIONAL:
        # Plot only the first subject
        # (otherwise 80+ windows will open)
        # -------------------------------------------------
        if file == selected_files[0]:
            proc.plot_ica_timeseries(num_components=10)
            plot_ica_topomaps_from_processor(proc, n_components=10)

        # -------------------------------------------------
        # STEP 3 — Remove detected ICA components
        # -------------------------------------------------
        proc.remove_artifacts(bad_components)

        # -------------------------------------------------
        # STEP 4 — Interpolate bad channels
        # -------------------------------------------------
        proc.interpolate_bad_channels()

        # -------------------------------------------------
        # OPTIONAL:
        # Plot only the first subject
        # -------------------------------------------------
        if file == selected_files[0]:
            proc.plot_eeg_cleaned(subject_name=file)
            proc.plot_psd(channel=0)
            proc.plot_before_after(channel=0)

    # -------------------------------------------------
    # Save ICA statistics
    # -------------------------------------------------

    import pandas as pd

    df = pd.DataFrame(component_summary)

    df.to_csv("ica_component_summary.csv", index=False)

    print("\n====================================")
    print("ICA SUMMARY")
    print("====================================")
    print(df.head())

    print(f"\nSubjects processed : {len(df)}")
    print(f"Average removed ICs: {df['n_removed'].mean():.2f}")
    print(f"Median removed ICs : {df['n_removed'].median():.2f}")
    print(f"Maximum removed ICs: {df['n_removed'].max()}")

    print("\nSaved:")
    print("ica_component_summary.csv")


if __name__ == "__main__":
    main()

## Medication-State Ground Truth Labels

PD-OFF / PD-ON are derived from the clinical `Visit_Meds_Status` table (see `data_processing.ipynb`), **not** from the `_1_`/`_2_` run number in the filename -- each PD patient was recorded once, entirely in one medication state; the two files per patient are two repeated runs of the same visit. Patient 803 has EEG data but no clinical medication label and is excluded from every cell below.

In [ ]:
import glob
import os
import re

import numpy as np
from scipy.stats import zscore

# --------------------------------------------------
# CLINICAL GROUND TRUTH: PD MEDICATION STATUS PER PATIENT
# --------------------------------------------------
# Source: data_processing.ipynb's create_dataset() clinical metadata table
# (Visit_Meds_Status column).
#
# IMPORTANT — this is a BETWEEN-SUBJECTS design, not a within-subject
# crossover: each PD patient was recorded during a single EEG visit, entirely
# in ONE medication state. The two files per patient ('..._1_...' and
# '..._2_...') are two repeated ~3-minute resting-state RUNS recorded
# back-to-back within that SAME visit -- not an OFF-then-ON session pair.
# PD-OFF and PD-ON are therefore two disjoint groups of patients (13 OFF,
# 14 ON), each compared against the same HC pool, rather than paired
# per-patient measurements. Every downstream grouped split in this notebook
# respects that: a patient's epochs never leak across train/test, but there
# is no "this patient's OFF matched to their own ON" comparison to be made.
PD_MEDICATION_STATUS = {
    801: 'ON', 802: 'OFF', 804: 'ON', 805: 'ON', 806: 'OFF', 807: 'OFF',
    808: 'OFF', 809: 'ON', 810: 'ON', 811: 'ON', 813: 'OFF', 814: 'ON',
    815: 'ON', 816: 'OFF', 817: 'OFF', 818: 'ON', 819: 'OFF', 820: 'ON',
    821: 'ON', 822: 'ON', 823: 'OFF', 824: 'OFF', 825: 'ON', 826: 'ON',
    827: 'OFF', 828: 'OFF', 829: 'OFF',
}

HC, PD_OFF, PD_ON = 0, 1, 2
CLASS_NAMES = {HC: 'HC', PD_OFF: 'PD-OFF', PD_ON: 'PD-ON'}


def get_medication_label(pid):
    """HC / PD_OFF / PD_ON, or None if `pid` has no clinical ground truth.

    Patient 803 has EEG epochs on disk but was never entered in the clinical
    metadata table -- it must be EXCLUDED rather than guessed at.
    """
    pid = int(pid)
    if pid in PD_MEDICATION_STATUS:
        return PD_OFF if PD_MEDICATION_STATUS[pid] == 'OFF' else PD_ON
    # HC: four-digit codes (e.g. 8010) or the 890-999 matched-control range,
    # per the patient ID convention used throughout the rest of this project.
    if pid >= 1000 or 890 <= pid <= 999:
        return HC
    return None


def extract_patient_id(filename):
    filename = os.path.basename(filename)
    match = re.match(r'^(\d+)_', filename)
    if match:
        return match.group(1)
    raise ValueError(f'Could not extract patient ID from filename: {filename}')


def load_medication_labeled_epochs(outlier_z=4.0, verbose=True):
    """Loads every epoch file and labels it HC / PD-OFF / PD-ON from the
    clinical ground truth (NOT from the '_1_'/'_2_' run number -- see the note
    above), dropping any patient without a ground-truth medication status.
    """
    epoch_files = sorted(glob.glob('epochs/*.npz'))
    epoch_files = [
        f for f in epoch_files
        if not re.search(r'REST\d+\.npz$', os.path.basename(f))
    ]
    if not epoch_files:
        raise FileNotFoundError("No EEG epoch files found in 'epochs/'")

    all_features, all_labels, all_patient_ids = [], [], []
    excluded_patients = set()

    for f in epoch_files:
        pid = extract_patient_id(f)
        label = get_medication_label(pid)
        if label is None:
            excluded_patients.add(pid)
            continue

        d = np.load(f, allow_pickle=True)
        X_file = d['features']
        all_features.append(X_file)
        all_labels.append(np.full(len(X_file), label, dtype=int))
        all_patient_ids.extend([pid] * len(X_file))

    if verbose and excluded_patients:
        print(
            'Excluded (no clinical medication ground truth): '
            f'{sorted(excluded_patients, key=int)}'
        )

    X = np.vstack(all_features)
    y = np.concatenate(all_labels)
    patient_ids = np.array(all_patient_ids)

    if verbose:
        print(f'\nLoaded : {X.shape}')
        for cls in (HC, PD_OFF, PD_ON):
            n_epochs = int(np.sum(y == cls))
            n_patients = len(set(patient_ids[y == cls]))
            print(f'  {CLASS_NAMES[cls]:<7} : {n_epochs:>6} epochs  ({n_patients} patients)')

    # Outlier removal -- same z<4 rule used throughout the rest of the project
    z = np.abs(zscore(X, axis=0, nan_policy='omit'))
    z = np.nan_to_num(z, nan=0.0)
    mask = (z < outlier_z).all(axis=1)
    X, y, patient_ids = X[mask], y[mask], patient_ids[mask]

    if verbose:
        print(f'After outlier removal (z<{outlier_z}) : {X.shape}')

    return X, y, patient_ids


# --------------------------------------------------
# GLOBAL POOL / HOLD-OUT PATIENT SPLIT
# --------------------------------------------------
# This split MUST run once, here, before any encoder (AE/CAE/VAE) or the
# conditional DDPM is ever trained -- NOT just before the final classifier.
#
# Why: if the encoders and DDPM are fit on every patient and only the
# downstream classifier later does a "clean" patient-grouped train/test
# split, the generative models have already been shaped by the patients that
# split calls "unseen test patients". A classifier trained on synthetic data
# sampled from that generator isn't testing generalization to unseen
# patients at all -- it's testing recall of patients the generator (if not
# the classifier) already saw. That's what was producing the suspiciously
# perfect TSTR scores (AUC=1.000, zero variance across 21 classifier/source
# combinations) found during review.
#
# The fix: carve out HOLDOUT_PATIENTS here, first. Every encoder/DDPM cell
# below filters to POOL patients only before training. The ML and downstream
# cells then use this exact same split as their test set, so the held-out
# patients are genuinely unseen by the ENTIRE pipeline, not just the final
# classifier.
HOLDOUT_FRACTION = 0.25
SPLIT_RANDOM_STATE = 42


def split_pool_and_holdout_patients(y, patient_ids, holdout_fraction=HOLDOUT_FRACTION,
                                     random_state=SPLIT_RANDOM_STATE):
    """Splits PATIENTS (not epochs) into a 'pool' set (used for everything --
    encoder training, DDPM training, AND classifier training) and a 'holdout'
    set (used ONLY as the final test set, touched by nothing upstream).

    Stratified per class (HC / PD-OFF / PD-ON) so the holdout set has
    patients from all three groups.
    """
    rng = np.random.default_rng(random_state)
    unique_patients, first_idx = np.unique(patient_ids, return_index=True)
    patient_class = y[first_idx]

    pool_patients, holdout_patients = [], []
    for cls in (HC, PD_OFF, PD_ON):
        cls_patients = unique_patients[patient_class == cls]
        n_holdout = max(1, int(round(len(cls_patients) * holdout_fraction)))
        shuffled = rng.permutation(cls_patients)
        holdout_patients.extend(shuffled[:n_holdout].tolist())
        pool_patients.extend(shuffled[n_holdout:].tolist())

    pool_patients, holdout_patients = set(pool_patients), set(holdout_patients)
    assert not (pool_patients & holdout_patients), 'HOLDOUT LEAKAGE DETECTED'

    return pool_patients, holdout_patients


def get_pool_and_holdout_masks(patient_ids, pool_patients, holdout_patients):
    """Boolean epoch-level masks matching the patient-level split above."""
    pool_mask = np.isin(patient_ids, list(pool_patients))
    holdout_mask = np.isin(patient_ids, list(holdout_patients))
    return pool_mask, holdout_mask


if __name__ == '__main__':
    X_all, y_all, patient_ids_all = load_medication_labeled_epochs()

    pool_patients, holdout_patients = split_pool_and_holdout_patients(y_all, patient_ids_all)
    pool_mask, holdout_mask = get_pool_and_holdout_masks(patient_ids_all, pool_patients, holdout_patients)

    print('\nGlobal pool / hold-out patient split (fixed for the rest of this notebook):')
    for cls in (HC, PD_OFF, PD_ON):
        unique_patients, first_idx = np.unique(patient_ids_all, return_index=True)
        patient_class = y_all[first_idx]
        cls_patients = set(unique_patients[patient_class == cls])
        print(f'  {CLASS_NAMES[cls]:<7} : {len(cls_patients & pool_patients)} pool patients, '
              f'{len(cls_patients & holdout_patients)} held out')
    print(f'\nPool epochs    : {int(pool_mask.sum())}')
    print(f'Holdout epochs : {int(holdout_mask.sum())}')
    print('\nHeld-out patients (never seen by any encoder, DDPM, or classifier training):',
          sorted(holdout_patients, key=int))


## Autoencoders

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# NOTE: every file this cell (and every other cell in this notebook) saves
# uses the '_med' suffix. This notebook shares its working directory with
# eeg_analysis_fixed.ipynb -- without the suffix, running this would silently
# overwrite that notebook's autoencoder_weights.pth / latent_autoencoder.npz /
# etc. (the plain binary HC-vs-PD pipeline).

HC, PD_OFF, PD_ON = 0, 1, 2
CLASS_NAMES = {HC: 'HC', PD_OFF: 'PD-OFF', PD_ON: 'PD-ON'}
CLASS_COLORS = {HC: 'steelblue', PD_OFF: 'tomato', PD_ON: 'gold'}


# --------------------------------------------------
# AUTOENCODER ARCHITECTURE (same as the single-pipeline notebook)
# --------------------------------------------------

class EEGAutoencoder(nn.Module):

  def __init__(self, input_dim, latent_dim):
    super().__init__()

    self.encoder = nn.Sequential(
        nn.Linear(input_dim, 256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.LeakyReLU(0.2),
        nn.Linear(64, latent_dim),
    )

    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, 64),
        nn.BatchNorm1d(64),
        nn.LeakyReLU(0.2),
        nn.Linear(64, 128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(128, 256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(256, input_dim),
    )

  def forward(self, x):
    z = self.encoder(x)
    x_hat = self.decoder(z)
    return x_hat, z

  def encode(self, x):
    return self.encoder(x)

  def decode(self, z):
    return self.decoder(z)


# --------------------------------------------------
# TRAINER (unchanged from the single-pipeline notebook -- the AE is
# unsupervised, so nothing about medication-state labels touches this class)
# --------------------------------------------------

class AutoencoderTrainer:

  def __init__(self, input_dim, latent_dim=22, lr=3e-4, batch_size=512, epochs=200):
    self.latent_dim = latent_dim
    self.batch_size = batch_size
    self.epochs = epochs
    self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    self.model = EEGAutoencoder(input_dim, latent_dim).to(self.device)
    self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-5)
    self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, patience=15, factor=0.5)
    self.criterion = nn.MSELoss()

    print(f'Device      : {self.device}')
    print(f'Input dim   : {input_dim}')
    print(f'Latent dim  : {latent_dim}')
    print(f'Parameters  : {sum(p.numel() for p in self.model.parameters()):,}')

  def fit(self, X):
    X_train, X_val = train_test_split(X, test_size=0.1, random_state=42)

    train_ds = TensorDataset(torch.FloatTensor(X_train))
    val_ds = TensorDataset(torch.FloatTensor(X_val))

    train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=self.batch_size)

    train_losses, val_losses = [], []
    best_val_loss = np.inf
    best_weights = None

    print(f'\nTraining autoencoder for {self.epochs} epochs...')
    print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'LR':>10}")
    print('-' * 48)

    for epoch in range(1, self.epochs + 1):
      self.model.train()
      train_loss = 0.0
      for (batch,) in train_loader:
        batch = batch.to(self.device)
        self.optimizer.zero_grad()
        x_hat, _ = self.model(batch)
        loss = self.criterion(x_hat, batch)
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        train_loss += loss.item() * len(batch)
      train_loss /= len(train_ds)

      self.model.eval()
      val_loss = 0.0
      with torch.no_grad():
        for (batch,) in val_loader:
          batch = batch.to(self.device)
          x_hat, _ = self.model(batch)
          val_loss += self.criterion(x_hat, batch).item() * len(batch)
      val_loss /= len(val_ds)

      self.scheduler.step(val_loss)
      train_losses.append(train_loss)
      val_losses.append(val_loss)

      if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}

      if epoch % 10 == 0 or epoch == 1:
        lr = self.optimizer.param_groups[0]['lr']
        print(f'{epoch:>6} | {train_loss:>12.6f} | {val_loss:>10.6f} | {lr:>10.2e}')

    self.model.load_state_dict(best_weights)
    print(f'\nBest val loss : {best_val_loss:.6f}')
    return train_losses, val_losses

  def encode_all(self, X):
    self.model.eval()
    with torch.no_grad():
      z = self.model.encode(torch.FloatTensor(X).to(self.device)).cpu().numpy()
    return z

  def decode_all(self, Z):
    self.model.eval()
    with torch.no_grad():
      x = self.model.decode(torch.FloatTensor(Z).to(self.device)).cpu().numpy()
    return x


# --------------------------------------------------
# PLOTS
# --------------------------------------------------

def plot_loss(train_losses, val_losses):
  plt.figure(figsize=(8, 4))
  plt.plot(train_losses, label='Train loss', lw=1.2)
  plt.plot(val_losses, label='Val loss', lw=1.2)
  plt.xlabel('Epoch')
  plt.ylabel('MSE loss')
  plt.title('Autoencoder training')
  plt.legend()
  plt.tight_layout()
  plt.show()


def plot_latent_space_3class(Z, y, title='Autoencoder latent space — HC vs PD-OFF vs PD-ON'):
  """2D PCA projection of the latent space, colored by the 3-way medication label."""
  from sklearn.decomposition import PCA

  pca = PCA(n_components=2, random_state=42)
  Z_2d = pca.fit_transform(Z)

  plt.figure(figsize=(7, 5))
  for cls in (HC, PD_OFF, PD_ON):
    mask = y == cls
    plt.scatter(
        Z_2d[mask, 0], Z_2d[mask, 1],
        label=f'{CLASS_NAMES[cls]} (n={mask.sum()})',
        alpha=0.3, s=8, color=CLASS_COLORS[cls],
    )
  plt.xlabel('PC1 of latent')
  plt.ylabel('PC2 of latent')
  plt.title(title)
  plt.legend()
  plt.tight_layout()
  plt.show()


def plot_reconstruction(trainer, X, n_samples=5):
  idx = np.random.choice(len(X), n_samples, replace=False)
  X_sub = X[idx]

  trainer.model.eval()
  with torch.no_grad():
    X_hat, _ = trainer.model(torch.FloatTensor(X_sub).to(trainer.device))
    X_hat = X_hat.cpu().numpy()

  fig, axes = plt.subplots(n_samples, 1, figsize=(12, 2.5 * n_samples))
  axes = np.atleast_1d(axes)
  for i, ax in enumerate(axes):
    ax.plot(X_sub[i], lw=0.8, label='Original', alpha=0.8)
    ax.plot(X_hat[i], lw=0.8, label='Reconstructed', alpha=0.8)
    ax.set_yticks([])
    ax.legend(fontsize=7, loc='upper right')
  plt.suptitle('Reconstruction check', fontsize=12)
  plt.tight_layout()
  plt.show()


# --------------------------------------------------
# MAIN
# --------------------------------------------------

if __name__ == '__main__':

  # load_medication_labeled_epochs() / split_pool_and_holdout_patients() are
  # defined in the labels cell (run it first). Labels are 3-class (HC=0 /
  # PD-OFF=1 / PD-ON=2) from the clinical Visit_Meds_Status table, not the
  # '_1_'/'_2_' run number.
  X_raw, y, patient_ids = load_medication_labeled_epochs()

  # CRITICAL: train the encoder on POOL patients only. The held-out patients
  # must stay unseen by every encoder/DDPM in this notebook, not just by the
  # final classifier -- otherwise "generalizes to unseen patients" claims
  # downstream are invalid (the generator will have already been shaped by
  # those patients' data). See the labels cell for the full explanation.
  pool_patients, holdout_patients = split_pool_and_holdout_patients(y, patient_ids)
  pool_mask, _ = get_pool_and_holdout_masks(patient_ids, pool_patients, holdout_patients)
  X_raw, y, patient_ids = X_raw[pool_mask], y[pool_mask], patient_ids[pool_mask]
  print(f'\nTraining on POOL patients only: {X_raw.shape} '
        f'({len(set(patient_ids))} of {len(set(patient_ids)) + len(holdout_patients)} patients; '
        f'{len(holdout_patients)} held out)')

  # ── Normalize feature space (pooled across all 3 classes) ────────────
  scaler = StandardScaler()
  X_clean = scaler.fit_transform(X_raw)

  # ── Train Autoencoder ─────────────────────────────────────────────────
  # Trained on the POOLED HC + PD-OFF + PD-ON data so every class shares one
  # latent space -- the conditional DDPM cell later splits PD-OFF vs PD-ON
  # within that shared space rather than needing two separate encoders.
  latent_dim = 22
  trainer = AutoencoderTrainer(
      input_dim=X_clean.shape[1],
      latent_dim=latent_dim,
      lr=3e-4,
      batch_size=512,
      epochs=200,
  )

  train_losses, val_losses = trainer.fit(X_clean)

  plot_loss(train_losses, val_losses)
  plot_reconstruction(trainer, X_clean)

  Z_raw = trainer.encode_all(X_clean)

  z_mean = np.mean(Z_raw, axis=0)
  z_std = np.std(Z_raw, axis=0) + 1e-8
  Z = (Z_raw - z_mean) / z_std

  print(f'\nLatent shape : {Z.shape}')
  print(f'Latent Mean  : {np.mean(Z):.4f} (expected ~0.0)')
  print(f'Latent Std   : {np.std(Z):.4f}  (expected ~1.0)')

  plot_latent_space_3class(Z, y)

  # ── Save artifacts (medication-suffixed filenames) ────────────────────
  np.savez('latent_autoencoder_med.npz', X_latent=Z, y=y, patient_ids=patient_ids, z_mean=z_mean, z_std=z_std)

  torch.save(trainer.model.state_dict(), 'autoencoder_weights_med.pth')
  torch.save(trainer.model.decoder.state_dict(), 'autoencoder_decoder_weights_med.pth')

  with open('ae_scaler_med.pkl', 'wb') as f:
    pickle.dump(scaler, f)

  print('\nSaved Successfully:')
  print(f'  latent_autoencoder_med.npz          — standardized latents {Z.shape} (3-class y) with z_mean/z_std')
  print('  autoencoder_weights_med.pth         — full model state_dict')
  print('  autoencoder_decoder_weights_med.pth — decoder state_dict for the conditional DDPM bio-loss')
  print('  ae_scaler_med.pkl                   — feature scaler for the inverse transform')


## Contrastive Autoencoder (Medication-Aware)

The supervised contrastive loss is unchanged from the binary pipeline -- it already generalizes to any number of classes -- but it now receives the 3-class HC / PD-OFF / PD-ON label, so it actively pulls PD-OFF and PD-ON epochs into separate clusters in the shared latent space instead of just separating PD from HC.

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

HC, PD_OFF, PD_ON = 0, 1, 2
CLASS_NAMES = {HC: 'HC', PD_OFF: 'PD-OFF', PD_ON: 'PD-ON'}
CLASS_COLORS = {HC: 'steelblue', PD_OFF: 'tomato', PD_ON: 'gold'}


# --------------------------------------------------
# 1. AUTOENCODER ARCHITECTURE (identical to the AE cell above)
# --------------------------------------------------

class EEGAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


# --------------------------------------------------
# 2. SUPERVISED CONTRASTIVE LOSS MODULE
# --------------------------------------------------
# Unchanged from the binary single-pipeline version. The mask here is built
# from torch.eq(y, y.T) with NO assumption about how many classes exist, so
# feeding it the 3-class label (HC / PD-OFF / PD-ON) instead of a binary one
# just works: it pulls same-medication-state epochs together and pushes
# HC / PD-OFF / PD-ON apart from each other in the shared latent space. That
# is actually a better fit for this notebook's goal than the binary version --
# it directly encourages the latent space to separate PD-OFF from PD-ON, which
# the downstream conditional DDPM and classifiers both rely on.

class SupervisedContrastiveAELoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.01, temperature=0.07):
        super().__init__()
        self.recon_criterion = nn.SmoothL1Loss(beta=0.1)
        self.alpha = alpha
        self.beta = beta
        self.temperature = temperature

    def forward(self, x_hat, x, z, y):
        recon_loss = self.recon_criterion(x_hat, x)

        z_norm = F.normalize(z, dim=1)
        sim_matrix = torch.matmul(z_norm, z_norm.T) / self.temperature

        y = y.view(-1, 1)
        mask = torch.eq(y, y.T).float().to(z.device)

        logits_mask = torch.ones_like(mask) - torch.eye(y.shape[0], device=z.device)
        mask = mask * logits_mask

        exp_sim = torch.exp(sim_matrix) * logits_mask
        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
        contrastive_loss = -mean_log_prob_pos.mean()

        total_loss = (self.alpha * recon_loss) + (self.beta * contrastive_loss)
        return total_loss, recon_loss, contrastive_loss


# --------------------------------------------------
# 3. CONTRASTIVE TRAINER
# --------------------------------------------------

class ContrastiveAutoencoderTrainer:
    def __init__(self, input_dim, latent_dim=22, lr=1e-3, batch_size=256, epochs=100, alpha=1.0, beta=0.1):
        self.latent_dim = latent_dim
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.model = EEGAutoencoder(input_dim, latent_dim).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-5)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, patience=15, factor=0.5)
        self.criterion = SupervisedContrastiveAELoss(alpha=alpha, beta=beta, temperature=0.07)

        print(f'Device      : {self.device}')
        print(f'Input dim   : {input_dim}')
        print(f'Latent dim  : {latent_dim}')
        print(f'Parameters  : {sum(p.numel() for p in self.model.parameters()):,}')

    def fit(self, X, y):
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.1, random_state=42, stratify=y
        )

        train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
        val_ds = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))

        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size)

        train_losses, val_losses = [], []
        best_val_loss = np.inf
        best_weights = None

        print(f'\nTraining Contrastive Autoencoder for {self.epochs} epochs...')
        print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'LR':>10}")
        print('-' * 48)

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            train_loss = 0.0
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                x_hat, z = self.model(batch_x)
                loss, _, _ = self.criterion(x_hat, batch_x, z, batch_y)

                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()

                train_loss += loss.item() * len(batch_x)
            train_loss /= len(train_ds)

            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x = batch_x.to(self.device)
                    batch_y = batch_y.to(self.device)
                    x_hat, z = self.model(batch_x)
                    loss, _, _ = self.criterion(x_hat, batch_x, z, batch_y)
                    val_loss += loss.item() * len(batch_x)
            val_loss /= len(val_ds)

            self.scheduler.step(val_loss)
            train_losses.append(train_loss)
            val_losses.append(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}

            if epoch % 10 == 0 or epoch == 1:
                lr = self.optimizer.param_groups[0]['lr']
                print(f"{epoch:>6} | {train_loss:>12.6f} | {val_loss:>10.6f} | {lr:>10.2e}")

        self.model.load_state_dict(best_weights)
        print(f"\nBest val loss : {best_val_loss:.6f}")
        return train_losses, val_losses

    def encode_all(self, X):
        self.model.eval()
        with torch.no_grad():
            z = self.model.encode(torch.FloatTensor(X).to(self.device)).cpu().numpy()
        return z

    def decode_all(self, Z):
        self.model.eval()
        with torch.no_grad():
            x = self.model.decode(torch.FloatTensor(Z).to(self.device)).cpu().numpy()
        return x


# --------------------------------------------------
# 4. PLOTS
# --------------------------------------------------

def plot_loss(train_losses, val_losses):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Train loss', lw=1.2)
    plt.plot(val_losses, label='Val loss', lw=1.2)
    plt.xlabel("Epoch")
    plt.ylabel("Contrastive AE Loss")
    plt.title("Contrastive Autoencoder Training")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_latent_space_3class(Z, y, title="Contrastive AE Latent Space — HC vs PD-OFF vs PD-ON"):
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2, random_state=42)
    Z_2d = pca.fit_transform(Z)

    plt.figure(figsize=(7, 5))
    for cls in (HC, PD_OFF, PD_ON):
        mask = y == cls
        plt.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                    label=f"{CLASS_NAMES[cls]} (n={mask.sum()})",
                    alpha=0.3, s=8, color=CLASS_COLORS[cls])
    plt.xlabel("PC1 of latent")
    plt.ylabel("PC2 of latent")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_reconstruction(trainer, X, n_samples=5):
    idx = np.random.choice(len(X), n_samples, replace=False)
    X_sub = X[idx]

    trainer.model.eval()
    with torch.no_grad():
        X_hat, _ = trainer.model(torch.FloatTensor(X_sub).to(trainer.device))
        X_hat = X_hat.cpu().numpy()

    fig, axes = plt.subplots(n_samples, 1, figsize=(12, 2.5 * n_samples))
    axes = np.atleast_1d(axes)
    for i, ax in enumerate(axes):
        ax.plot(X_sub[i], lw=0.8, label='Original', alpha=0.8)
        ax.plot(X_hat[i], lw=0.8, label='Reconstructed', alpha=0.8)
        ax.set_yticks([])
        ax.legend(fontsize=7, loc='upper right')
    plt.suptitle("Reconstruction Check", fontsize=12)
    plt.tight_layout()
    plt.show()


# --------------------------------------------------
# 5. MAIN EXECUTION PIPELINE
# --------------------------------------------------

if __name__ == "__main__":

    X_raw, y, patient_ids = load_medication_labeled_epochs()

    # CRITICAL: train on POOL patients only -- see the labels cell's note on
    # why holding patients out only at the final classifier isn't enough.
    pool_patients, holdout_patients = split_pool_and_holdout_patients(y, patient_ids)
    pool_mask, _ = get_pool_and_holdout_masks(patient_ids, pool_patients, holdout_patients)
    X_raw, y, patient_ids = X_raw[pool_mask], y[pool_mask], patient_ids[pool_mask]
    print(f"\nTraining on POOL patients only: {X_raw.shape} "
          f"({len(set(patient_ids))} of {len(set(patient_ids)) + len(holdout_patients)} patients; "
          f"{len(holdout_patients)} held out)")

    scaler = StandardScaler()
    X_clean = scaler.fit_transform(X_raw)

    trainer = ContrastiveAutoencoderTrainer(
        input_dim=X_clean.shape[1],
        latent_dim=22,
        lr=3e-4,
        batch_size=512,
        epochs=200,
        alpha=1.0,
        beta=0.005,
    )

    train_losses, val_losses = trainer.fit(X_clean, y)

    plot_loss(train_losses, val_losses)
    plot_reconstruction(trainer, X_clean)

    Z = trainer.encode_all(X_clean)
    print(f"\nLatent shape : {Z.shape}")

    plot_latent_space_3class(Z, y)

    # CAE latent has needs_znorm=False by convention elsewhere in this
    # project (its contrastive loss already keeps ||z|| well-behaved via the
    # cosine similarity, so no extra standardization is applied here).
    np.savez('latent_contrastive_med.npz', Z=Z, X_latent=Z, y=y, patient_ids=patient_ids)

    torch.save(trainer.model.decoder, 'contrastive_autoencoder_decoder_med.pth')
    torch.save(trainer.model.decoder.state_dict(), 'contrastive_decoder_weights_med.pth')
    torch.save(trainer.model.state_dict(), 'contrastive_autoencoder_weights_med.pth')

    with open('cae_scaler_med.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    print("\n✓ Saved Contrastive Autoencoder Artifacts:")
    print("  latent_contrastive_med.npz              — Latent vectors (N, 22), 3-class y, for the conditional DDPM")
    print("  contrastive_autoencoder_decoder_med.pth — Full decoder module for the conditional DDPM bio-loss")
    print("  contrastive_decoder_weights_med.pth     — Decoder state dictionary backup")
    print("  contrastive_autoencoder_weights_med.pth — Full CAE state dictionary")
    print("  cae_scaler_med.pkl                      — Feature scaler")


## Variational Autoencoder (VAE)

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

HC, PD_OFF, PD_ON = 0, 1, 2
CLASS_NAMES = {HC: 'HC', PD_OFF: 'PD-OFF', PD_ON: 'PD-ON'}
CLASS_COLORS = {HC: 'steelblue', PD_OFF: 'tomato', PD_ON: 'gold'}


# --------------------------------------------------
# 1. VARIATIONAL AUTOENCODER ARCHITECTURE
# --------------------------------------------------

class EEGVariationalAutoencoder(nn.Module):
  """Gaussian VAE over the 315-D band-power feature vector.

  The encoder trunk is the same stack the plain AE and the contrastive AE use,
  but it terminates in two heads (mu, logvar) instead of one latent projection.

  The decoder is kept ARCHITECTURALLY IDENTICAL to EEGAutoencoder.decoder on
  purpose: every downstream loader in this notebook (DDPM bio-loss, evaluation,
  machine-learning cells) rebuilds that exact nn.Sequential, so the decoder
  state_dict saved here drops straight into them with no special-casing.
  """

  def __init__(self, input_dim, latent_dim):
    super().__init__()

    # Encoder trunk: input_dim -> 64
    self.encoder = nn.Sequential(
        nn.Linear(input_dim, 256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.LeakyReLU(0.2),
    )

    # Two heads parameterising q(z|x) = N(mu, diag(exp(logvar)))
    self.fc_mu = nn.Linear(64, latent_dim)
    self.fc_logvar = nn.Linear(64, latent_dim)

    # Decoder: latent_dim -> input_dim  (same layer-for-layer as the AE/CAE)
    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, 64),
        nn.BatchNorm1d(64),
        nn.LeakyReLU(0.2),
        nn.Linear(64, 128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(128, 256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.2),
        nn.Linear(256, input_dim),
    )

  def encode(self, x):
    """Returns (mu, logvar). logvar is clamped for numerical safety."""
    h = self.encoder(x)
    mu = self.fc_mu(h)
    logvar = torch.clamp(self.fc_logvar(h), min=-10.0, max=10.0)
    return mu, logvar

  def reparameterize(self, mu, logvar):
    std = torch.exp(0.5 * logvar)
    return mu + std * torch.randn_like(std)

  def decode(self, z):
    return self.decoder(z)

  def forward(self, x):
    mu, logvar = self.encode(x)
    z = self.reparameterize(mu, logvar)
    x_hat = self.decode(z)
    return x_hat, mu, logvar, z


# --------------------------------------------------
# 2. ELBO LOSS (beta-VAE WITH FREE BITS)
# --------------------------------------------------

class VAELoss(nn.Module):
  """Negative ELBO = reconstruction + beta * KL[q(z|x) || N(0, I)].

  `free_bits` reserves a minimum KL budget per latent dimension. Without it a
  315-D -> 22-D VAE with a decoder this strong tends to posterior-collapse
  (logvar -> 0, mu -> 0) and the latent becomes pure noise, which would make
  the downstream DDPM useless.
  """

  def __init__(self, free_bits=0.1):
    super().__init__()
    self.free_bits = free_bits

  def forward(self, x_hat, x, mu, logvar, beta):
    # Reconstruction: summed over features, averaged over the batch, so it
    # stays on the same scale as the summed KL term.
    recon_loss = F.mse_loss(x_hat, x, reduction='none').sum(dim=1).mean()

    # Analytic KL against a standard normal prior, per latent dimension
    kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
    kl_dim_mean = kl_per_dim.mean(dim=0)  # (latent_dim,)

    kl_free = torch.clamp(kl_dim_mean, min=self.free_bits).sum()
    kl_raw = kl_dim_mean.sum()

    total_loss = recon_loss + beta * kl_free
    return total_loss, recon_loss, kl_raw, kl_dim_mean.detach()


# --------------------------------------------------
# 3. VAE TRAINER
# --------------------------------------------------

class VAETrainer:

  def __init__(
      self,
      input_dim,
      latent_dim=22,
      lr=3e-4,
      batch_size=512,
      epochs=200,
      beta_max=0.5,
      warmup_frac=0.3,
      free_bits=0.1,
  ):
    self.latent_dim = latent_dim
    self.batch_size = batch_size
    self.epochs = epochs
    self.beta_max = beta_max
    self.warmup_epochs = max(1, int(epochs * warmup_frac))
    self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    self.model = EEGVariationalAutoencoder(input_dim, latent_dim).to(self.device)
    self.optimizer = torch.optim.Adam(
        self.model.parameters(), lr=lr, weight_decay=1e-5
    )
    self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        self.optimizer, patience=15, factor=0.5
    )
    self.criterion = VAELoss(free_bits=free_bits)

    print(f'Device      : {self.device}')
    print(f'Input dim   : {input_dim}')
    print(f'Latent dim  : {latent_dim}')
    print(f'Beta (max)  : {beta_max}  (KL warm-up over {self.warmup_epochs} epochs)')
    print(f'Free bits   : {free_bits} nats/dim')
    print(f'Parameters  : {sum(p.numel() for p in self.model.parameters()):,}')

  def _beta_at(self, epoch):
    """Linear KL annealing: 0 -> beta_max over the warm-up window."""
    return self.beta_max * min(1.0, epoch / self.warmup_epochs)

  def fit(self, X):
    """Train the VAE on X (n_samples, n_features).

    Returns (train_losses, val_losses, kl_history).
    """
    X_train, X_val = train_test_split(X, test_size=0.1, random_state=42)

    train_ds = TensorDataset(torch.FloatTensor(X_train))
    val_ds = TensorDataset(torch.FloatTensor(X_val))

    train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=self.batch_size)

    train_losses, val_losses, kl_history = [], [], []
    best_val_loss = np.inf
    best_weights = None

    print(f'\nTraining variational autoencoder for {self.epochs} epochs...')
    print(
        f"{'Epoch':>6} | {'Beta':>6} | {'Train ELBO':>12} | {'Val ELBO':>10} |"
        f" {'Recon':>10} | {'KL':>8}"
    )
    print('-' * 70)

    for epoch in range(1, self.epochs + 1):
      beta = self._beta_at(epoch)

      # -- Train --------------------------------------------------
      self.model.train()
      train_loss = 0.0
      for (batch,) in train_loader:
        batch = batch.to(self.device)
        self.optimizer.zero_grad()
        x_hat, mu, logvar, _ = self.model(batch)
        loss, _, _, _ = self.criterion(x_hat, batch, mu, logvar, beta)
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        train_loss += loss.item() * len(batch)
      train_loss /= len(train_ds)

      # -- Validate -----------------------------------------------
      self.model.eval()
      val_loss = 0.0
      val_recon = 0.0
      val_kl = 0.0
      with torch.no_grad():
        for (batch,) in val_loader:
          batch = batch.to(self.device)
          x_hat, mu, logvar, _ = self.model(batch)
          loss, recon, kl, _ = self.criterion(x_hat, batch, mu, logvar, beta)
          val_loss += loss.item() * len(batch)
          val_recon += recon.item() * len(batch)
          val_kl += kl.item() * len(batch)
      val_loss /= len(val_ds)
      val_recon /= len(val_ds)
      val_kl /= len(val_ds)

      self.scheduler.step(val_loss)
      train_losses.append(train_loss)
      val_losses.append(val_loss)
      kl_history.append(val_kl)

      # Only start tracking "best" once beta has fully warmed up, otherwise
      # the cheapest epochs (beta ~ 0) always win and we keep a model whose
      # latent isn't regularised at all.
      if epoch >= self.warmup_epochs and val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {
            k: v.cpu().clone() for k, v in self.model.state_dict().items()
        }

      if epoch % 10 == 0 or epoch == 1:
        print(
            f'{epoch:>6} | {beta:>6.3f} | {train_loss:>12.4f} |'
            f' {val_loss:>10.4f} | {val_recon:>10.4f} | {val_kl:>8.4f}'
        )

    if best_weights is not None:
      self.model.load_state_dict(best_weights)
    print(f'\nBest val ELBO : {best_val_loss:.4f}')
    return train_losses, val_losses, kl_history

  def encode_all(self, X, sample=False):
    """Encode the dataset to latent space.

    Uses the posterior MEAN by default -- that is the deterministic, lowest
    noise representation and the one the DDPM should model. Set sample=True to
    draw z ~ q(z|x) instead.
    """
    self.model.eval()
    with torch.no_grad():
      x_t = torch.FloatTensor(X).to(self.device)
      mu, logvar = self.model.encode(x_t)
      z = self.model.reparameterize(mu, logvar) if sample else mu
    return z.cpu().numpy()

  def encode_stats(self, X):
    """Returns (mu, logvar) for the whole dataset -- used for diagnostics."""
    self.model.eval()
    with torch.no_grad():
      mu, logvar = self.model.encode(torch.FloatTensor(X).to(self.device))
    return mu.cpu().numpy(), logvar.cpu().numpy()

  def decode_all(self, Z):
    """Decode latent vectors back to feature space."""
    self.model.eval()
    with torch.no_grad():
      x = self.model.decode(torch.FloatTensor(Z).to(self.device)).cpu().numpy()
    return x

  def sample_prior(self, n_samples):
    """Ancestral sampling from the VAE prior: z ~ N(0, I) -> decoder.

    This is the VAE's own generative path. The DDPM trained further down does
    the same job but learns the actual aggregate posterior instead of assuming
    it matches the prior, which is usually the better generator -- this method
    is the baseline to compare it against.
    """
    self.model.eval()
    with torch.no_grad():
      z = torch.randn(n_samples, self.latent_dim, device=self.device)
      x = self.model.decode(z).cpu().numpy()
    return x


# --------------------------------------------------
# 4. PLOTS
# --------------------------------------------------

def plot_vae_loss(train_losses, val_losses, kl_history):
  fig, axes = plt.subplots(1, 2, figsize=(12, 4))

  axes[0].plot(train_losses, label='Train ELBO', lw=1.2)
  axes[0].plot(val_losses, label='Val ELBO', lw=1.2)
  axes[0].set_xlabel('Epoch')
  axes[0].set_ylabel('Negative ELBO')
  axes[0].set_title('VAE training')
  axes[0].legend()

  axes[1].plot(kl_history, color='mediumpurple', lw=1.2)
  axes[1].set_xlabel('Epoch')
  axes[1].set_ylabel('KL[q(z|x) || N(0,I)]')
  axes[1].set_title('KL divergence (watch for collapse to 0)')

  plt.tight_layout()
  plt.show()


def plot_vae_latent_space(Z, y, title='VAE latent space — HC vs PD-OFF vs PD-ON'):
  """2D PCA projection of the standardized latent space, colored by the
  3-way medication label."""
  from sklearn.decomposition import PCA

  pca = PCA(n_components=2, random_state=42)
  Z_2d = pca.fit_transform(Z)

  plt.figure(figsize=(7, 5))
  for cls in (HC, PD_OFF, PD_ON):
    mask = y == cls
    plt.scatter(
        Z_2d[mask, 0],
        Z_2d[mask, 1],
        label=f'{CLASS_NAMES[cls]} (n={mask.sum()})',
        alpha=0.3,
        s=8,
        color=CLASS_COLORS[cls],
    )
  plt.xlabel('PC1 of latent')
  plt.ylabel('PC2 of latent')
  plt.title(title)
  plt.legend()
  plt.tight_layout()
  plt.show()


def plot_vae_reconstruction(trainer, X, n_samples=5):
  """Compare original vs reconstructed features for a few samples."""
  idx = np.random.choice(len(X), n_samples, replace=False)
  X_sub = X[idx]

  trainer.model.eval()
  with torch.no_grad():
    x_hat, _, _, _ = trainer.model(torch.FloatTensor(X_sub).to(trainer.device))
    x_hat = x_hat.cpu().numpy()

  fig, axes = plt.subplots(n_samples, 1, figsize=(12, 2.5 * n_samples))
  axes = np.atleast_1d(axes)
  for i, ax in enumerate(axes):
    ax.plot(X_sub[i], lw=0.8, label='Original', alpha=0.8)
    ax.plot(x_hat[i], lw=0.8, label='Reconstructed', alpha=0.8)
    ax.set_yticks([])
    ax.legend(fontsize=7, loc='upper right')
  plt.suptitle('VAE reconstruction check', fontsize=12)
  plt.tight_layout()
  plt.show()


def report_latent_usage(kl_dim_mean, threshold=0.01):
  """Posterior-collapse diagnostic: how many latent dims carry information."""
  active = np.sum(kl_dim_mean > threshold)
  print(f'\nActive latent dimensions (KL > {threshold} nats): '
        f'{active}/{len(kl_dim_mean)}')
  print(f'Per-dim KL (sorted desc): '
        f'{np.round(np.sort(kl_dim_mean)[::-1], 3).tolist()}')
  if active < len(kl_dim_mean) * 0.5:
    print('  [WARNING] More than half the latent dimensions have collapsed. '
          'Lower beta_max or raise free_bits and retrain.')

  plt.figure(figsize=(8, 3.5))
  plt.bar(np.arange(len(kl_dim_mean)), kl_dim_mean, color='mediumpurple',
          alpha=0.85)
  plt.axhline(threshold, color='tomato', ls='--', lw=1, label='collapse threshold')
  plt.xlabel('Latent dimension')
  plt.ylabel('KL (nats)')
  plt.title('Per-dimension KL — latent usage')
  plt.legend()
  plt.tight_layout()
  plt.show()

  return active


# --------------------------------------------------
# 5. MAIN EXECUTION PIPELINE
# --------------------------------------------------

if __name__ == '__main__':

  # load_medication_labeled_epochs() is defined in an earlier cell. It
  # returns 3-class labels (HC=0 / PD-OFF=1 / PD-ON=2) from the clinical
  # Visit_Meds_Status table and already applies the z<4 outlier filter.
  X_raw, y, patient_ids = load_medication_labeled_epochs()

  # CRITICAL: train on POOL patients only -- see the labels cell's note on
  # why holding patients out only at the final classifier isn't enough.
  pool_patients, holdout_patients = split_pool_and_holdout_patients(y, patient_ids)
  pool_mask, _ = get_pool_and_holdout_masks(patient_ids, pool_patients, holdout_patients)
  X_raw, y, patient_ids = X_raw[pool_mask], y[pool_mask], patient_ids[pool_mask]
  print(f'\nTraining on POOL patients only: {X_raw.shape} '
        f'({len(set(patient_ids))} of {len(set(patient_ids)) + len(holdout_patients)} patients; '
        f'{len(holdout_patients)} held out)')

  # -- Normalize feature space (pooled across all 3 classes) -----------
  scaler = StandardScaler()
  X_clean = scaler.fit_transform(X_raw)

  # -- Train VAE -------------------------------------------------------
  latent_dim = 22
  vae_trainer = VAETrainer(
      input_dim=X_clean.shape[1],  # 315
      latent_dim=latent_dim,
      lr=3e-4,
      batch_size=512,
      epochs=200,
      beta_max=0.5,
      warmup_frac=0.3,
      free_bits=0.1,
  )

  train_losses, val_losses, kl_history = vae_trainer.fit(X_clean)

  plot_vae_loss(train_losses, val_losses, kl_history)
  plot_vae_reconstruction(vae_trainer, X_clean)

  # -- Encode to latent space & standardize ---------------------------
  mu_all, logvar_all = vae_trainer.encode_stats(X_clean)

  # Per-dimension KL over the whole dataset, for the collapse diagnostic
  kl_dim = np.mean(
      -0.5 * (1.0 + logvar_all - mu_all**2 - np.exp(logvar_all)), axis=0
  )
  n_active = report_latent_usage(kl_dim)

  Z_raw = mu_all  # posterior mean is the latent the DDPM will model

  # CRITICAL FOR DDPM: center and scale Z so mean=0.0 and std=1.0
  z_mean = np.mean(Z_raw, axis=0)
  z_std = np.std(Z_raw, axis=0) + 1e-8
  Z = (Z_raw - z_mean) / z_std

  print(f'\nLatent shape : {Z.shape}')
  print(f'Latent Mean  : {np.mean(Z):.4f} (expected ~0.0)')
  print(f'Latent Std   : {np.std(Z):.4f}  (expected ~1.0)')

  plot_vae_latent_space(Z, y)

  # -- Save artifacts (medication-suffixed filenames) -------------------
  np.savez('latent_vae_med.npz', X_latent=Z, Z=Z, y=y, patient_ids=patient_ids, z_mean=z_mean, z_std=z_std)

  torch.save(vae_trainer.model.state_dict(), 'vae_weights_med.pth')
  torch.save(vae_trainer.model.decoder.state_dict(), 'vae_decoder_weights_med.pth')
  torch.save(vae_trainer.model.decoder, 'vae_decoder_med.pth')

  with open('vae_scaler_med.pkl', 'wb') as f:
    pickle.dump(scaler, f)

  print('\nSaved Successfully:')
  print(f'  latent_vae_med.npz          — standardized latents {Z.shape} (3-class y) with z_mean/z_std')
  print('  vae_weights_med.pth         — full VAE state_dict (encoder + heads + decoder)')
  print('  vae_decoder_weights_med.pth — decoder-only state_dict for the conditional DDPM bio-loss')
  print('  vae_decoder_med.pth         — decoder as a serialized nn.Module')
  print('  vae_scaler_med.pkl          — 315-D feature scaler for the inverse transform')

  # -- Prior-sampling sanity check -------------------------------------
  # The VAE can generate on its own by decoding z ~ N(0, I). This is the
  # baseline the DDPM (next section) has to beat.
  print('\nPrior-sampling sanity check (VAE generative path, no DDPM):')
  X_prior_scaled = vae_trainer.sample_prior(1000)
  X_prior_raw = np.clip(scaler.inverse_transform(X_prior_scaled), 0.0, None)

  print(f'  Real raw feature mean        : {np.mean(X_raw):.4f}')
  print(f'  VAE prior-sample raw mean    : {np.mean(X_prior_raw):.4f}')
  print(f'  Real raw feature std         : {np.std(X_raw):.4f}')
  print(f'  VAE prior-sample raw std     : {np.std(X_prior_raw):.4f}')
  print(f'  Active latent dims           : {n_active}/{latent_dim}')


## Class-Conditional Diffusion Model — PD-OFF vs PD-ON

One diffusion model per encoder (AE / CAE / VAE), conditioned on the medication-state class rather than trained as two separate checkpoints -- see the note at the top of the next cell for why. `generate(n, class_label)` samples PD-OFF and PD-ON separately from the same trained network.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# --------------------------------------------------
# WHY A SINGLE CLASS-CONDITIONAL MODEL INSTEAD OF TWO SEPARATE DDPMs
# --------------------------------------------------
# The AE / CAE / VAE cells above already train ONE shared encoder on the
# pooled HC + PD-OFF + PD-ON data, so PD-OFF and PD-ON already live in the
# SAME latent space. Splitting the downstream generator into two independent
# DDPM checkpoints would throw that shared geometry away for no benefit here
# (both classes have latent vectors from the exact same encoder, on the exact
# same scale). Instead this cell trains ONE conditional DDPM per encoder,
# with the medication state (OFF=0 / ON=1) fed in as a class-embedding
# alongside the usual diffusion timestep embedding -- so `generate(n, class_label)`
# samples PD-OFF and PD-ON from the same trained network.
#
# Only the PD epochs are modeled here (HC is real and plentiful -- 4,222+
# epochs -- so there's no need to synthesize it).

HC, PD_OFF, PD_ON = 0, 1, 2


# --------------------------------------------------
# 1. NOISE SCHEDULE (identical to the single-pipeline DDPM)
# --------------------------------------------------

def make_beta_schedule(T=1000, beta_start=1e-4, beta_end=0.02):
  betas = torch.linspace(beta_start, beta_end, T)
  alphas = 1.0 - betas
  alphas_bar = torch.cumprod(alphas, dim=0)
  return betas, alphas, alphas_bar


def q_sample(x0, t, alphas_bar, noise=None):
  if noise is None:
    noise = torch.randn_like(x0)
  ab_t = alphas_bar[t].view(-1, 1)
  x_t = torch.sqrt(ab_t) * x0 + torch.sqrt(1.0 - ab_t) * noise
  return x_t, noise


# --------------------------------------------------
# 2. BIOPHYSICAL LOSS HELPER (unchanged -- 1/f power-law regularization)
# --------------------------------------------------

def compute_biophysical_loss(x_hat, device):
  """x_hat shape: (batch, 315) -> 63 channels * 5 frequency bands
  [Delta, Theta, Alpha, Beta, Tremor]. Enforces a 1/f log-power vs
  log-frequency line."""
  batch_size = x_hat.shape[0]
  x_reshaped = x_hat.view(batch_size, 63, 5)
  x_positive = F.softplus(x_reshaped) + 1e-6
  log_p = torch.log(x_positive)

  center_freqs = torch.tensor([2.0, 6.0, 10.0, 20.0, 40.0], device=device)
  log_f = torch.log(center_freqs)

  x_mean = log_f.mean()
  y_mean = log_p.mean(dim=-1, keepdim=True)
  x_dev = log_f - x_mean
  y_dev = log_p - y_mean

  slope = (y_dev * x_dev).sum(dim=-1, keepdim=True) / (x_dev**2).sum()
  intercept = y_mean - slope * x_mean
  log_p_target = slope * log_f + intercept

  return F.mse_loss(log_p, log_p_target)


# --------------------------------------------------
# 3. CLASS-CONDITIONAL DENOISING NETWORK
# --------------------------------------------------

class ConditionalDenoisingNetwork(nn.Module):
  """Same MLP backbone as the single-pipeline DenoisingNetwork, plus one
  extra nn.Embedding for the medication-state class. The class embedding is
  ADDED to the timestep embedding (the standard class-conditioning trick used
  in conditional diffusion models) rather than concatenated, so the backbone
  input width doesn't change and existing hyperparameters still apply."""

  def __init__(self, latent_dim, num_classes=2, T=1000, hidden_dim=256):
    super().__init__()

    self.time_embed = nn.Sequential(
        nn.Embedding(T, hidden_dim),
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU(),
    )
    self.class_embed = nn.Embedding(num_classes, hidden_dim)

    self.net = nn.Sequential(
        nn.Linear(latent_dim + hidden_dim, 512),
        nn.LayerNorm(512),
        nn.SiLU(),
        nn.Dropout(0.1),
        nn.Linear(512, 512),
        nn.LayerNorm(512),
        nn.SiLU(),
        nn.Dropout(0.1),
        nn.Linear(512, 256),
        nn.LayerNorm(256),
        nn.SiLU(),
        nn.Linear(256, latent_dim),
    )

  def forward(self, x_t, t, c):
    t_emb = self.time_embed(t)
    c_emb = self.class_embed(c)
    cond = t_emb + c_emb
    x_in = torch.cat([x_t, cond], dim=-1)
    return self.net(x_in)


# --------------------------------------------------
# 4. CONDITIONAL DDPM TRAINER
# --------------------------------------------------

class ConditionalDDPMTrainer:

  def __init__(
      self,
      latent_dim,
      num_classes=2,
      T=1000,
      lr=2e-4,
      batch_size=256,
      epochs=500,
      decoder_model=None,
      scaler=None,
      lambda_bio=0.05,
  ):
    self.T = T
    self.batch_size = batch_size
    self.epochs = epochs
    self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    self.decoder_model = decoder_model
    self.lambda_bio = lambda_bio

    if self.decoder_model is not None:
      self.decoder_model = self.decoder_model.to(self.device)
      self.decoder_model.eval()
      for p in self.decoder_model.parameters():
        p.requires_grad = False

    if scaler is not None:
      mean_val = np.array(scaler.mean_, dtype=np.float32)
      scale_val = np.array(scaler.scale_, dtype=np.float32)
      self.scaler_mean = torch.tensor(mean_val, dtype=torch.float32, device=self.device)
      self.scaler_scale = torch.tensor(scale_val, dtype=torch.float32, device=self.device)
    else:
      self.scaler_mean, self.scaler_scale = None, None

    self.betas, self.alphas, self.alphas_bar = make_beta_schedule(T)
    self.betas = self.betas.to(self.device)
    self.alphas = self.alphas.to(self.device)
    self.alphas_bar = self.alphas_bar.to(self.device)

    self.model = ConditionalDenoisingNetwork(latent_dim, num_classes, T).to(self.device)
    self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
    self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=epochs)
    self.criterion = nn.MSELoss()

    print(f'Device      : {self.device}')
    print(f'Latent dim  : {latent_dim}')
    print(f'Classes     : {num_classes} (0=PD-OFF, 1=PD-ON)')
    print(f'T steps     : {T}')
    print(f'Bio-Penalty : lambda={self.lambda_bio if self.decoder_model is not None else "Disabled"}')
    print(f'Parameters  : {sum(p.numel() for p in self.model.parameters()):,}')

  def fit(self, Z, class_labels):
    """Z: (N, latent_dim) PD latents (OFF + ON pooled). class_labels: (N,)
    ints in {0, 1} -- 0=PD-OFF, 1=PD-ON."""
    dataset = TensorDataset(torch.FloatTensor(Z), torch.LongTensor(class_labels))
    loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

    losses = []
    print(f'\nTraining conditional DDPM on {len(Z)} PD epochs '
          f'(OFF={int(np.sum(class_labels==0))}, ON={int(np.sum(class_labels==1))})...')
    print(f"{'Epoch':>6} | {'Total Loss':>12} | {'Noise Loss':>12} | {'Bio Loss':>12}")
    print('-' * 52)

    for epoch in range(1, self.epochs + 1):
      self.model.train()
      epoch_loss = 0.0
      epoch_noise_loss = 0.0
      epoch_bio_loss = 0.0

      for batch, c_batch in loader:
        batch = batch.to(self.device)
        c_batch = c_batch.to(self.device)
        B = len(batch)

        t = torch.randint(0, self.T, (B,), device=self.device)
        x_t, noise = q_sample(batch, t, self.alphas_bar)

        noise_pred = self.model(x_t, t, c_batch)
        loss_simple = self.criterion(noise_pred, noise)

        if self.decoder_model is not None:
          ab_t = self.alphas_bar[t].view(-1, 1)
          sqrt_ab_t = torch.sqrt(torch.clamp(ab_t, min=1e-4))
          sqrt_one_minus_ab_t = torch.sqrt(1.0 - ab_t)

          z0_hat_raw = (x_t - sqrt_one_minus_ab_t * noise_pred) / sqrt_ab_t
          z0_hat = torch.clamp(z0_hat_raw, -4.0, 4.0)

          # NOTE (inherited from the single-pipeline DDPM): this only undoes
          # THIS trainer's own per-source latent StandardScaler, not the
          # encoder's separate z_mean/z_std latent normalization from the
          # AE/VAE cells. The decoder therefore sees a slightly different
          # scale here than it was trained on. The bio-loss is a soft,
          # low-weight (lambda_bio=0.05) regularizer, not the primary
          # training signal, so this is a pre-existing quirk carried over
          # rather than a new bug -- flagged here so it isn't silently
          # mysterious if you go looking.
          if self.scaler_mean is not None and self.scaler_scale is not None:
            z0_hat_unscaled = (z0_hat * self.scaler_scale) + self.scaler_mean
          else:
            z0_hat_unscaled = z0_hat

          x0_hat = self.decoder_model(z0_hat_unscaled)
          loss_bio = compute_biophysical_loss(x0_hat, self.device)
          loss = loss_simple + (self.lambda_bio * loss_bio)
          epoch_bio_loss += loss_bio.item() * B
        else:
          loss = loss_simple

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        epoch_loss += loss.item() * B
        epoch_noise_loss += loss_simple.item() * B

      epoch_loss /= len(Z)
      epoch_noise_loss /= len(Z)
      epoch_bio_loss /= len(Z)

      self.scheduler.step()
      losses.append(epoch_loss)

      if epoch % 50 == 0 or epoch == 1:
        if self.decoder_model is not None:
          print(f'{epoch:>6} | {epoch_loss:>12.6f} | {epoch_noise_loss:>12.6f} | {epoch_bio_loss:>12.6f}')
        else:
          print(f'{epoch:>6} | {epoch_loss:>12.6f} | {epoch_noise_loss:>12.6f} | {"N/A":>12}')

    print(f'\nFinal loss : {losses[-1]:.6f}')
    return losses

  @torch.no_grad()
  def generate(self, n_samples, class_label):
    """class_label: int, 0=PD-OFF or 1=PD-ON. Samples n_samples latents
    conditioned on that single class."""
    self.model.eval()
    latent_dim = self.model.net[-1].out_features
    x = torch.randn(n_samples, latent_dim, device=self.device)
    c = torch.full((n_samples,), class_label, dtype=torch.long, device=self.device)

    for t_val in reversed(range(self.T)):
      t_batch = torch.full((n_samples,), t_val, device=self.device, dtype=torch.long)
      noise_pred = self.model(x, t_batch, c)

      alpha = self.alphas[t_val]
      alpha_bar = self.alphas_bar[t_val]
      beta = self.betas[t_val]

      mean = (1 / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1 - alpha_bar)) * noise_pred)

      if t_val > 0:
        z = torch.randn_like(x)
        x = mean + torch.sqrt(beta) * z
      else:
        x = mean

    return x.cpu().numpy()


# --------------------------------------------------
# 5. ROBUST DECODER LOADER (same pattern used throughout this project)
# --------------------------------------------------

class GenericDecoder(nn.Module):
  """Matches the decoder architecture shared by the AE / CAE / VAE cells."""

  def __init__(self, latent_dim=22, input_dim=315):
    super().__init__()
    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
        nn.Linear(64, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
        nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
        nn.Linear(256, input_dim),
    )

  def forward(self, z):
    return self.decoder(z)


def load_decoder(decoder_path):
  """Loads a decoder saved as a full nn.Module, a full-model state_dict, or a
  decoder-only state_dict (bare keys) -- covers every format the AE/CAE/VAE
  cells in this notebook save."""
  if decoder_path is None or not os.path.exists(decoder_path):
    return None
  try:
    loaded_data = torch.load(decoder_path, map_location='cpu')
    if isinstance(loaded_data, nn.Module):
      return loaded_data.eval()
    elif isinstance(loaded_data, dict):
      model = GenericDecoder()
      keys = list(loaded_data.keys())
      if any(k.startswith('decoder.') or k.startswith('encoder.') for k in keys):
        model.load_state_dict(loaded_data, strict=False)
      else:
        model.decoder.load_state_dict(loaded_data)
      return model.eval()
  except Exception as e:
    print(f"⚠️ Warning: Failed to load decoder '{decoder_path}': {e}")
    return None


# --------------------------------------------------
# 6. MAIN EXECUTION -- ONE CONDITIONAL DDPM PER ENCODER
# --------------------------------------------------

if __name__ == '__main__':

  SOURCES = [
      ('latent_autoencoder_med.npz', 'AE',
       ['autoencoder_decoder_weights_med.pth'], 'ae_scaler_med.pkl'),
      ('latent_contrastive_med.npz', 'CAE',
       ['contrastive_decoder_weights_med.pth', 'contrastive_autoencoder_decoder_med.pth'], 'cae_scaler_med.pkl'),
      ('latent_vae_med.npz', 'VAE',
       ['vae_decoder_weights_med.pth', 'vae_decoder_med.pth'], 'vae_scaler_med.pkl'),
  ]

  for latent_file, name, decoder_files, feature_scaler_file in SOURCES:
    if not os.path.exists(latent_file):
      print(f"\nSkipping {name}: '{latent_file}' not found.")
      continue

    print(f"\n{'='*60}")
    print(f'Training class-conditional DDPM on {name} latent space (PD-OFF vs PD-ON)')
    print(f"{'='*60}")

    d = np.load(latent_file)
    Z_all = d['X_latent'] if 'X_latent' in d else d['Z']
    y_all = d['y']

    pd_mask = y_all != HC
    Z_pd = Z_all[pd_mask]
    y_pd = y_all[pd_mask]
    class_labels = np.where(y_pd == PD_ON, 1, 0)  # local encoding: 0=OFF, 1=ON

    n_off = int(np.sum(class_labels == 0))
    n_on = int(np.sum(class_labels == 1))
    print(f'PD-OFF epochs : {n_off}')
    print(f'PD-ON  epochs : {n_on}')

    decoder_path = next((f for f in decoder_files if os.path.exists(f)), None)
    decoder = load_decoder(decoder_path)
    if decoder is None:
      print(f'  [INFO] No decoder found for {name} -- training without the biophysical loss term.')

    ddpm_scaler = StandardScaler()
    Z_pd_scaled = ddpm_scaler.fit_transform(Z_pd)

    trainer = ConditionalDDPMTrainer(
        latent_dim=Z_pd.shape[1],
        num_classes=2,
        T=1000,
        lr=2e-4,
        batch_size=64,
        epochs=500,
        decoder_model=decoder,
        scaler=ddpm_scaler,
        lambda_bio=0.05,
    )

    losses = trainer.fit(Z_pd_scaled, class_labels)

    plt.figure(figsize=(7, 4))
    plt.plot(losses, lw=1.2)
    plt.xlabel('Epoch')
    plt.ylabel('Total loss')
    plt.title(f'Conditional DDPM training — {name}')
    plt.tight_layout()
    plt.show()

    print(f'\nGenerating {n_off} synthetic PD-OFF and {n_on} synthetic PD-ON samples for {name}...')
    Z_off_syn_scaled = trainer.generate(n_samples=n_off, class_label=0)
    Z_on_syn_scaled = trainer.generate(n_samples=n_on, class_label=1)

    Z_off_syn = ddpm_scaler.inverse_transform(Z_off_syn_scaled)
    Z_on_syn = ddpm_scaler.inverse_transform(Z_on_syn_scaled)

    off_file = f'synthetic_pd_off_{name.lower()}_med.npz'
    on_file = f'synthetic_pd_on_{name.lower()}_med.npz'
    np.savez(off_file, Z_synthetic=Z_off_syn, Z=Z_off_syn)
    np.savez(on_file, Z_synthetic=Z_on_syn, Z=Z_on_syn)
    print(f"✓ Saved synthetic PD-OFF latents to '{off_file}'")
    print(f"✓ Saved synthetic PD-ON  latents to '{on_file}'")

    ddpm_latent_scaler_file = f'{name.lower()}_ddpm_latent_scaler_med.pkl'
    with open(ddpm_latent_scaler_file, 'wb') as f:
      pickle.dump(ddpm_scaler, f)
    print(f"✓ Saved DDPM latent scaler to '{ddpm_latent_scaler_file}'")

    ddpm_model_path = f'ddpm_cond_{name.lower()}_med.pth'
    torch.save(trainer.model.state_dict(), ddpm_model_path)
    print(f"✓ Saved conditional DDPM weights to '{ddpm_model_path}'")


## Evaluation — Biophysical Validation (Beta-Band Suppression)

Statistics / band-power / MMD comparisons run once per medication state, plus the headline validation: does each synthetic source reproduce the Beta-band (13-30 Hz) OFF-vs-ON direction actually observed in the real cohort? (Framed against the real cohort's own observed direction, not the textbook L-Dopa-suppression assumption -- see the cell for why, given this is a 13-vs-14-patient between-subjects cohort.)

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy import stats
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler

HC, PD_OFF, PD_ON = 0, 1, 2
CLASS_NAMES = {HC: 'HC', PD_OFF: 'PD-OFF', PD_ON: 'PD-ON'}

# Band order matches feat_names in epochs/*.npz: [Delta, Theta, Alpha,
# Beta (Motor), Tremor] repeated per channel (63 channels x 5 bands = 315).
BANDS = {'Delta': (0.5, 4), 'Theta': (4, 8), 'Alpha': (8, 13), 'Beta': (13, 30), 'Tremor': (3, 6)}
N_BANDS = len(BANDS)
BAND_NAMES = list(BANDS.keys())
BETA_IDX = BAND_NAMES.index('Beta')
SRATE = 500


# --------------------------------------------------
# DECODER LOADING (robust to full-model / decoder-only state dicts)
# --------------------------------------------------

class GenericDecoder(nn.Module):
  def __init__(self, latent_dim=22, input_dim=315):
    super().__init__()
    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
        nn.Linear(64, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
        nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
        nn.Linear(256, input_dim),
    )

  def forward(self, z):
    return self.decoder(z)


def load_decoder_safely(decoder_path, latent_dim=22, output_dim=315):
  loaded_data = torch.load(decoder_path, map_location='cpu')
  if isinstance(loaded_data, nn.Module):
    model = loaded_data
  elif isinstance(loaded_data, dict):
    model = GenericDecoder(latent_dim=latent_dim, input_dim=output_dim)
    keys = list(loaded_data.keys())
    if any(k.startswith('decoder.') or k.startswith('encoder.') for k in keys):
      model.load_state_dict(loaded_data, strict=False)
    else:
      model.decoder.load_state_dict(loaded_data)
  else:
    raise ValueError(f'Unknown data type inside {decoder_path}: {type(loaded_data)}')
  model.eval()
  return model


# --------------------------------------------------
# DECODE ONE SOURCE'S SYNTHETIC OFF/ON LATENTS BACK TO 315-D
# --------------------------------------------------

# name -> (decoder file candidates, encoder latent-znorm meta file, needs_znorm,
#          315-D feature scaler file)
SOURCE_CONFIGS = {
    'AE': (['autoencoder_decoder_weights_med.pth'], 'latent_autoencoder_med.npz', True, 'ae_scaler_med.pkl'),
    'CAE': (['contrastive_decoder_weights_med.pth', 'contrastive_autoencoder_decoder_med.pth'],
             'latent_contrastive_med.npz', False, 'cae_scaler_med.pkl'),
    'VAE': (['vae_decoder_weights_med.pth', 'vae_decoder_med.pth'], 'latent_vae_med.npz', True, 'vae_scaler_med.pkl'),
}


def decode_medication_synthetic(name, state_tag):
  """state_tag: 'off' or 'on'. Returns a 315-D raw-feature array, or None if
  the required files aren't present (DDPM/encoder cells not yet run for this
  source)."""
  decoder_files, meta_file, needs_znorm, feature_scaler_file = SOURCE_CONFIGS[name]
  syn_file = f'synthetic_pd_{state_tag}_{name.lower()}_med.npz'
  decoder_file = next((f for f in decoder_files if os.path.exists(f)), None)

  if not (os.path.exists(syn_file) and decoder_file):
    return None

  data = np.load(syn_file)
  # The conditional-DDPM cell already calls ddpm_scaler.inverse_transform
  # before saving synthetic_pd_*_med.npz, so Z here is already back in the
  # ENCODER's own latent scale (the DDPM's own per-source scaler needs no
  # further undoing) -- only the encoder's latent z-normalization (Step 1
  # below) still needs undoing before decoding.
  Z_syn = data['Z_synthetic'] if 'Z_synthetic' in data else data['Z']

  # Step 1: undo the encoder's latent z-normalization (22-D), for AE/VAE.
  if needs_znorm:
    if not os.path.exists(meta_file):
      print(f"  [WARNING] {meta_file} not found -- decoding {name} without latent z-renorm.")
    else:
      meta = np.load(meta_file)
      if 'z_mean' in meta and 'z_std' in meta and meta['z_mean'].shape[0] == Z_syn.shape[1]:
        Z_syn = (Z_syn * meta['z_std']) + meta['z_mean']
      else:
        print(f"  [WARNING] {meta_file} latent dim mismatch -- decoding {name} without latent z-renorm.")

  # Step 2: decode (22-D latent -> 315-D scaled feature space)
  decoder = load_decoder_safely(decoder_file)
  with torch.no_grad():
    X_scaled = decoder(torch.FloatTensor(Z_syn)).numpy()

  # Step 3: undo the 315-D raw-feature scaler, AFTER decoding.
  X_raw = X_scaled
  if os.path.exists(feature_scaler_file):
    with open(feature_scaler_file, 'rb') as f:
      feature_scaler = pickle.load(f)
    scale_ref = getattr(feature_scaler, 'scale_', getattr(feature_scaler, 'mean_', None))
    if scale_ref is not None and scale_ref.shape[0] == X_scaled.shape[1]:
      X_raw = feature_scaler.inverse_transform(X_scaled)
    else:
      print(f"  [WARNING] '{feature_scaler_file}' dimension mismatch for {name} -- returning scaled-space features.")
  else:
    print(f"  [WARNING] No 315-D feature scaler found for {name} -- returning scaled-space features.")

  return np.clip(X_raw, a_min=1e-6, a_max=None)


def load_real_and_synthetic_medication():
  """Loads real HC / PD-OFF / PD-ON epochs and every available source's
  decoded synthetic PD-OFF / PD-ON."""
  X, y, patient_ids = load_medication_labeled_epochs()

  X_real_hc = X[y == HC]
  X_real_off = X[y == PD_OFF]
  X_real_on = X[y == PD_ON]

  print(f'Real HC     : {X_real_hc.shape}')
  print(f'Real PD-OFF : {X_real_off.shape}')
  print(f'Real PD-ON  : {X_real_on.shape}')

  synthetics_off, synthetics_on = {}, {}
  for name in SOURCE_CONFIGS:
    X_off = decode_medication_synthetic(name, 'off')
    X_on = decode_medication_synthetic(name, 'on')
    if X_off is not None:
      synthetics_off[f'Synth {name}'] = X_off
      print(f'► Loaded Synth {name} PD-OFF: {X_off.shape}')
    else:
      print(f'⚠️ Skipped Synth {name} PD-OFF (files missing -- run the conditional DDPM cell first)')
    if X_on is not None:
      synthetics_on[f'Synth {name}'] = X_on
      print(f'► Loaded Synth {name} PD-ON : {X_on.shape}')
    else:
      print(f'⚠️ Skipped Synth {name} PD-ON  (files missing -- run the conditional DDPM cell first)')

  return X_real_hc, X_real_off, X_real_on, synthetics_off, synthetics_on


# --------------------------------------------------
# 1. GENERIC DISTRIBUTIONAL METRICS (statistics / band power / correlation / MMD)
# --------------------------------------------------
# Reused verbatim from the single-pipeline evaluation cell, called once per
# medication state below.

def compare_statistics(label_real, X_real, synthetics):
  print(f'\n{"="*60}\n1. BASIC STATISTICS — {label_real}\n{"="*60}')
  for name, data in [(f'Real {label_real}', X_real)] + list(synthetics.items()):
    flat = data.flatten()
    print(f'\n{name}:')
    print(f'  Mean={np.mean(flat):.4f}  Std={np.std(flat):.4f}  Median={np.median(flat):.4f}')


def compare_band_power(label_real, X_real, synthetics):
  print(f'\n{"="*60}\n2. BAND POWER COMPARISON — {label_real}\n{"="*60}')
  results = {
      b: {f'Real {label_real}': np.mean(X_real[:, list(range(i, X_real.shape[1], N_BANDS))])}
      for i, b in enumerate(BAND_NAMES)
  }
  for syn_name, syn_data in synthetics.items():
    for i, b in enumerate(BAND_NAMES):
      results[b][syn_name] = np.mean(syn_data[:, list(range(i, syn_data.shape[1], N_BANDS))])

  for b in BAND_NAMES:
    real_p = results[b][f'Real {label_real}']
    print(f'\n{b}:  Real={real_p:.6f}')
    for syn_name in synthetics:
      syn_p = results[b][syn_name]
      err = abs(syn_p - real_p) / (abs(real_p) + 1e-8) * 100
      print(f'  {syn_name:<10} : {syn_p:.6f}  (error: {err:.1f}%)')
  return results


def compare_mmd(label_real, X_real, synthetics, sample_size=500):
  print(f'\n{"="*60}\n3. MMD SCORE — {label_real} (lower = more similar)\n{"="*60}')
  mmd_scores = {}
  for syn_name, syn_data in synthetics.items():
    idx_r = np.random.choice(len(X_real), min(sample_size, len(X_real)), replace=False)
    idx_s = np.random.choice(len(syn_data), min(sample_size, len(syn_data)), replace=False)
    R, S = X_real[idx_r], syn_data[idx_s]

    sc = StandardScaler()
    R_norm, S_norm = sc.fit_transform(R), sc.transform(S)
    all_samples = np.vstack([R_norm, S_norm])
    median_dist = np.median(pdist(all_samples, metric='euclidean'))
    gamma = 1.0 / (2 * (median_dist**2) + 1e-8)

    Kxx = rbf_kernel(R_norm, R_norm, gamma=gamma)
    Kyy = rbf_kernel(S_norm, S_norm, gamma=gamma)
    Kxy = rbf_kernel(R_norm, S_norm, gamma=gamma)
    score = float(Kxx.mean() + Kyy.mean() - 2 * Kxy.mean())
    mmd_scores[syn_name] = score
    print(f'  MMD (Real {label_real} vs {syn_name:<10}) : {score:.6f}')
  return mmd_scores


def compare_wasserstein(label_real, X_real, synthetics):
  """Per-feature Wasserstein (earth-mover's) distance, averaged over all 315
  features. Unlike MMD (a kernel-space score with no natural units),
  Wasserstein distance is in the same units as the feature itself, so it can
  be sanity-checked directly against e.g. the band-power comparison above."""
  from scipy.stats import wasserstein_distance

  print(f'\n{"="*60}\n4. WASSERSTEIN DISTANCE — {label_real} (lower = more similar)\n{"="*60}')
  wasserstein_scores = {}
  n_features = X_real.shape[1]
  for syn_name, syn_data in synthetics.items():
    per_feature = [
        wasserstein_distance(X_real[:, j], syn_data[:, j])
        for j in range(n_features)
    ]
    mean_w = float(np.mean(per_feature))
    wasserstein_scores[syn_name] = mean_w
    print(f'  Wasserstein (Real {label_real} vs {syn_name:<10}) : {mean_w:.6f}  '
          f'(mean over {n_features} features)')
  return wasserstein_scores


def compare_kl_divergence(label_real, X_real, synthetics, bins=50):
  """Per-feature KL(Real || Synthetic), estimated via shared histogram
  binning -- there's no closed form from samples alone, so this is an
  approximation and genuinely sensitive to the bin count (bins=50 is a
  reasonable default for the hundreds-to-thousands of epochs per class here,
  but don't treat the exact value as more precise than that). Add-one
  smoothing avoids log(0)/divide-by-zero when a bin is empty in one
  distribution but not the other."""
  from scipy.stats import entropy

  print(f'\n{"="*60}\n5. KL DIVERGENCE — {label_real} (lower = more similar; '
        f'KL(Real || Synthetic))\n{"="*60}')
  kl_scores = {}
  n_features = X_real.shape[1]
  for syn_name, syn_data in synthetics.items():
    per_feature = []
    for j in range(n_features):
      real_col, syn_col = X_real[:, j], syn_data[:, j]
      bin_edges = np.histogram_bin_edges(np.concatenate([real_col, syn_col]), bins=bins)
      p_hist, _ = np.histogram(real_col, bins=bin_edges)
      q_hist, _ = np.histogram(syn_col, bins=bin_edges)
      p = (p_hist + 1) / (p_hist.sum() + len(p_hist))
      q = (q_hist + 1) / (q_hist.sum() + len(q_hist))
      per_feature.append(entropy(p, q))
    mean_kl = float(np.mean(per_feature))
    kl_scores[syn_name] = mean_kl
    print(f'  KL div (Real {label_real} || {syn_name:<10}) : {mean_kl:.6f}  '
          f'(mean over {n_features} features, {bins} bins)')
  return kl_scores


# --------------------------------------------------
# 2. BETA-BAND SUPPRESSION CHECK (the headline medication-state validation)
# --------------------------------------------------
# L-Dopa is well established to suppress motor-cortex Beta-band (13-30 Hz)
# power. This is a BETWEEN-SUBJECTS cohort here (PD-OFF and PD-ON are two
# disjoint groups of patients, not the same patients on/off meds -- see the
# labeling cell), so the group-level Beta difference is a noisier signal than
# a paired within-subject design would give, and at n=13 vs n=14 patients it
# may not even reach significance in the REAL data. Rather than assert the
# textbook direction as a hard pass/fail, this checks whether each synthetic
# source's OFF-vs-ON Beta direction agrees with the direction actually
# observed in the REAL data -- that is the honest bar the generator has to
# clear given this cohort.

def beta_band_suppression_check(X_real_off, X_real_on, synthetics_off, synthetics_on):
  print(f'\n{"="*70}\nBETA-BAND (13-30 Hz) SUPPRESSION UNDER MEDICATION\n{"="*70}')

  beta_cols = list(range(BETA_IDX, X_real_off.shape[1], N_BANDS))

  def per_epoch_beta(X):
    return X[:, beta_cols].mean(axis=1)

  beta_real_off = per_epoch_beta(X_real_off)
  beta_real_on = per_epoch_beta(X_real_on)

  u_stat, p_val = stats.mannwhitneyu(beta_real_off, beta_real_on, alternative='two-sided')
  real_direction = 'OFF > ON' if beta_real_off.mean() > beta_real_on.mean() else 'ON > OFF'

  print(f'\nReal PD-OFF Beta power : {beta_real_off.mean():.4f} ± {beta_real_off.std():.4f}  (n={len(beta_real_off)} epochs)')
  print(f'Real PD-ON  Beta power : {beta_real_on.mean():.4f} ± {beta_real_on.std():.4f}  (n={len(beta_real_on)} epochs)')
  print(f'Observed direction     : {real_direction}  (Mann-Whitney U p={p_val:.4g})')
  print('Classical L-Dopa literature predicts OFF > ON (Beta suppressed by medication);')
  print('this is a between-subjects cohort of 13 OFF / 14 ON patients, so treat the')
  print('observed direction above as this cohort\'s ground truth to reproduce, not')
  print('as a foregone conclusion.')

  rows = []
  for name in synthetics_off:
    if name not in synthetics_on:
      continue
    beta_syn_off = per_epoch_beta(synthetics_off[name])
    beta_syn_on = per_epoch_beta(synthetics_on[name])
    syn_direction = 'OFF > ON' if beta_syn_off.mean() > beta_syn_on.mean() else 'ON > OFF'
    agrees = syn_direction == real_direction

    print(f'\n{name}:')
    print(f'  Synth PD-OFF Beta : {beta_syn_off.mean():.4f} ± {beta_syn_off.std():.4f}')
    print(f'  Synth PD-ON  Beta : {beta_syn_on.mean():.4f} ± {beta_syn_on.std():.4f}')
    print(f'  Direction         : {syn_direction}  '
          f'({"✓ matches real cohort" if agrees else "✗ DISAGREES with real cohort"})')

    rows.append({
        'source': name,
        'real_off_beta': beta_real_off.mean(), 'real_on_beta': beta_real_on.mean(),
        'real_direction': real_direction, 'real_p_value': p_val,
        'synth_off_beta': beta_syn_off.mean(), 'synth_on_beta': beta_syn_on.mean(),
        'synth_direction': syn_direction, 'direction_agrees': agrees,
    })

  if rows:
    fig, ax = plt.subplots(figsize=(1.8 * (len(rows) + 1), 4.5))
    labels = ['Real'] + [r['source'] for r in rows]
    off_vals = [beta_real_off.mean()] + [r['synth_off_beta'] for r in rows]
    on_vals = [beta_real_on.mean()] + [r['synth_on_beta'] for r in rows]
    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width / 2, off_vals, width, label='PD-OFF', color='tomato', alpha=0.85)
    ax.bar(x + width / 2, on_vals, width, label='PD-ON', color='gold', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylabel('Mean Beta-band power')
    ax.set_title('Beta-band power: PD-OFF vs PD-ON, real and synthetic')
    ax.legend()
    plt.tight_layout()
    plt.show()

  return rows


# --------------------------------------------------
# MAIN EXECUTION
# --------------------------------------------------

if __name__ == '__main__':
  print('Initializing medication-state comparative evaluation...')
  X_real_hc, X_real_off, X_real_on, synthetics_off, synthetics_on = load_real_and_synthetic_medication()

  if not synthetics_off and not synthetics_on:
    print('\n❌ No synthetic PD-OFF/PD-ON data found. Run the conditional DDPM cell first.')
  else:
    distributional_rows = []

    if synthetics_off:
      compare_statistics('PD-OFF', X_real_off, synthetics_off)
      band_off = compare_band_power('PD-OFF', X_real_off, synthetics_off)
      mmd_off = compare_mmd('PD-OFF', X_real_off, synthetics_off)
      wasserstein_off = compare_wasserstein('PD-OFF', X_real_off, synthetics_off)
      kl_off = compare_kl_divergence('PD-OFF', X_real_off, synthetics_off)
      for src in synthetics_off:
        distributional_rows.append({
            'state': 'PD-OFF', 'source': src,
            'MMD': mmd_off[src], 'Wasserstein': wasserstein_off[src],
            'KL_divergence': kl_off[src],
        })
    if synthetics_on:
      compare_statistics('PD-ON', X_real_on, synthetics_on)
      band_on = compare_band_power('PD-ON', X_real_on, synthetics_on)
      mmd_on = compare_mmd('PD-ON', X_real_on, synthetics_on)
      wasserstein_on = compare_wasserstein('PD-ON', X_real_on, synthetics_on)
      kl_on = compare_kl_divergence('PD-ON', X_real_on, synthetics_on)
      for src in synthetics_on:
        distributional_rows.append({
            'state': 'PD-ON', 'source': src,
            'MMD': mmd_on[src], 'Wasserstein': wasserstein_on[src],
            'KL_divergence': kl_on[src],
        })

    beta_rows = beta_band_suppression_check(X_real_off, X_real_on, synthetics_off, synthetics_on)

    if beta_rows:
      import pandas as pd
      pd.DataFrame(beta_rows).to_csv('medication_beta_suppression_results.csv', index=False)
      print('\nSaved -> medication_beta_suppression_results.csv')

    if distributional_rows:
      import pandas as pd
      pd.DataFrame(distributional_rows).to_csv('medication_distributional_metrics.csv', index=False)
      print('Saved -> medication_distributional_metrics.csv')


## Machine Learning — Medication-State Augmented Classification

Part A: 3-class HC / PD-OFF / PD-ON classification, real vs synthetic-augmented. Part B: a dedicated binary PD-OFF vs PD-ON "medication-state" classifier -- detecting whether a patient has taken their medication.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

HC, PD_OFF, PD_ON = 0, 1, 2
CLASS_NAMES = {HC: 'HC', PD_OFF: 'PD-OFF', PD_ON: 'PD-ON'}
RANDOM_STATE = 42


# --------------------------------------------------
# 1. DECODE SYNTHETIC OFF/ON DATA (same approach as the evaluation cell,
#    duplicated here so this cell can run standalone)
# --------------------------------------------------

class GenericDecoder(nn.Module):
    def __init__(self, latent_dim=22, input_dim=315):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(256, input_dim),
        )

    def forward(self, z):
        return self.decoder(z)


def load_decoder_safely(decoder_path, latent_dim=22, output_dim=315):
    loaded_data = torch.load(decoder_path, map_location="cpu")
    if isinstance(loaded_data, nn.Module):
        model = loaded_data
    elif isinstance(loaded_data, dict):
        model = GenericDecoder(latent_dim=latent_dim, input_dim=output_dim)
        keys = list(loaded_data.keys())
        if any(k.startswith("decoder.") or k.startswith("encoder.") for k in keys):
            model.load_state_dict(loaded_data, strict=False)
        else:
            model.decoder.load_state_dict(loaded_data)
    else:
        raise ValueError(f"Unknown data type inside {decoder_path}: {type(loaded_data)}")
    model.eval()
    return model


SOURCE_CONFIGS = {
    "AE": (["autoencoder_decoder_weights_med.pth"], "latent_autoencoder_med.npz", True, "ae_scaler_med.pkl"),
    "CAE": (["contrastive_decoder_weights_med.pth", "contrastive_autoencoder_decoder_med.pth"],
             "latent_contrastive_med.npz", False, "cae_scaler_med.pkl"),
    "VAE": (["vae_decoder_weights_med.pth", "vae_decoder_med.pth"], "latent_vae_med.npz", True, "vae_scaler_med.pkl"),
}


def decode_medication_synthetic(name, state_tag):
    decoder_files, meta_file, needs_znorm, feature_scaler_file = SOURCE_CONFIGS[name]
    syn_file = f"synthetic_pd_{state_tag}_{name.lower()}_med.npz"
    decoder_file = next((f for f in decoder_files if os.path.exists(f)), None)

    if not (os.path.exists(syn_file) and decoder_file):
        return None

    data = np.load(syn_file)
    Z_syn = data["Z_synthetic"] if "Z_synthetic" in data else data["Z"]

    if needs_znorm and os.path.exists(meta_file):
        meta = np.load(meta_file)
        if "z_mean" in meta and "z_std" in meta and meta["z_mean"].shape[0] == Z_syn.shape[1]:
            Z_syn = (Z_syn * meta["z_std"]) + meta["z_mean"]

    decoder = load_decoder_safely(decoder_file)
    with torch.no_grad():
        X_scaled = decoder(torch.FloatTensor(Z_syn)).numpy()

    X_raw = X_scaled
    if os.path.exists(feature_scaler_file):
        with open(feature_scaler_file, "rb") as f:
            feature_scaler = pickle.load(f)
        scale_ref = getattr(feature_scaler, "scale_", getattr(feature_scaler, "mean_", None))
        if scale_ref is not None and scale_ref.shape[0] == X_scaled.shape[1]:
            X_raw = feature_scaler.inverse_transform(X_scaled)

    return np.clip(X_raw, a_min=1e-6, a_max=None)


def load_synthetic_medication():
    synthetics_off, synthetics_on = {}, {}
    for name in SOURCE_CONFIGS:
        X_off = decode_medication_synthetic(name, "off")
        X_on = decode_medication_synthetic(name, "on")
        if X_off is not None:
            synthetics_off[name] = X_off
            print(f"Synth {name} PD-OFF : {X_off.shape}")
        if X_on is not None:
            synthetics_on[name] = X_on
            print(f"Synth {name} PD-ON  : {X_on.shape}")
    return synthetics_off, synthetics_on


# --------------------------------------------------
# 1b. CAE ENCODER (full encoder+decoder, not just the decoder used above) --
#     for classifying directly in the CAE's own 22-D latent space.
# --------------------------------------------------
# WHY: the CAE's supervised contrastive loss explicitly optimizes its 22-D
# latent space to separate HC / PD-OFF / PD-ON. Every experiment above
# decodes synthetic CAE latents back out to noisy 315-D raw features before
# classifying -- throwing away exactly the structure the CAE was trained to
# build. This block tests the alternative: encode REAL epochs into that same
# latent space (via the CAE's own encoder) and classify there directly,
# using the synthetic latents in their native 22-D form (no decode step).

class CAEEncoderDecoder(nn.Module):
    """Matches cell_cae.py's EEGAutoencoder exactly -- needed here because
    loading the encoder requires the full architecture, not just the
    GenericDecoder's decoder-only half used elsewhere in this cell."""

    def __init__(self, input_dim=315, latent_dim=22):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(256, input_dim),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


def load_cae_encoder(weights_path="contrastive_autoencoder_weights_med.pth"):
    """Loads the FULL CAE (encoder+decoder) so .encode() is usable. Returns
    None if the full-model weights aren't present (only the decoder-only
    checkpoint is required elsewhere in this cell, so this is optional)."""
    if not os.path.exists(weights_path):
        return None
    state_dict = torch.load(weights_path, map_location="cpu")
    model = CAEEncoderDecoder()
    model.load_state_dict(state_dict)
    model.eval()
    return model


def encode_with_cae(X_raw, cae_model, cae_scaler):
    """Raw 315-D features -> CAE's native 22-D latent space."""
    X_scaled = cae_scaler.transform(X_raw)
    with torch.no_grad():
        Z = cae_model.encode(torch.FloatTensor(X_scaled)).numpy()
    return Z


# --------------------------------------------------
# 2. TRAIN/TEST SPLIT -- REUSES THE GLOBAL POOL/HOLDOUT SPLIT
# --------------------------------------------------
# IMPORTANT: this must be the SAME split used to decide which patients the
# AE/CAE/VAE encoders and the conditional DDPM were trained on
# (split_pool_and_holdout_patients(), defined in the labels cell). Re-deriving
# an independent "fresh" patient split here -- even a correctly patient-
# grouped one -- would test the classifier against patients the GENERATOR
# had already seen, which is the leak that produced the suspiciously perfect
# TSTR scores (AUC=1.000, zero variance) found during review. Using the exact
# same pool/holdout partition everywhere is what makes "holdout" actually
# mean unseen by the whole pipeline, not just by this one classifier.

def patient_level_split_3class(X, y, patient_ids, random_state=SPLIT_RANDOM_STATE):
    """Thin wrapper around the canonical pool/holdout split: pool -> "train",
    holdout -> "test". Kept under this name so the rest of this cell's
    variable names (X_train/X_test/...) don't need to change."""
    pool_patients, holdout_patients = split_pool_and_holdout_patients(
        y, patient_ids, random_state=random_state
    )
    pool_mask, holdout_mask = get_pool_and_holdout_masks(patient_ids, pool_patients, holdout_patients)
    assert not (set(patient_ids[pool_mask]) & set(patient_ids[holdout_mask])), "PATIENT LEAKAGE DETECTED"

    return (X[pool_mask], y[pool_mask], patient_ids[pool_mask],
            X[holdout_mask], y[holdout_mask], patient_ids[holdout_mask])


def balance_3class(X, y, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = min(np.sum(y == c) for c in (HC, PD_OFF, PD_ON))
    idxs = [rng.choice(np.where(y == c)[0], size=n, replace=False) for c in (HC, PD_OFF, PD_ON)]
    sel = np.concatenate(idxs)
    rng.shuffle(sel)
    return X[sel], y[sel]


def augment_with_synthetic_3class(X_train, y_train, X_syn_off, X_syn_on, random_state=RANDOM_STATE):
    """Tops PD-OFF and PD-ON up toward the LARGEST real class count using
    synthetic epochs, so the augmented training set is roughly balanced
    across all three classes.

    Deliberately NOT hard-coded to "top up toward HC" (that was this
    function's first draft): in the single-pipeline binary notebook HC is
    always the plentiful class, but once PD is split into OFF/ON, either
    subgroup can individually be as large as or larger than HC (it is, in
    this cohort -- HC=1667 epochs vs OFF=1871 vs ON=2222). Targeting a fixed
    class instead of whichever is actually largest would make `top_up` a
    silent no-op whenever that fixed class isn't the largest -- exactly what
    happened before this fix (every "+Synthetic" experiment produced
    byte-identical results because no synthetic data was ever added)."""
    rng = np.random.default_rng(random_state)
    n_hc = int(np.sum(y_train == HC))

    off_real = X_train[y_train == PD_OFF]
    on_real = X_train[y_train == PD_ON]

    target_n = max(n_hc, len(off_real), len(on_real))

    def top_up(real, synth, target_n):
        if len(real) >= target_n or synth is None or len(synth) == 0:
            return real
        n_needed = min(target_n - len(real), len(synth))
        idx = rng.choice(len(synth), size=n_needed, replace=False)
        return np.vstack([real, synth[idx]])

    off_aug = top_up(off_real, X_syn_off, target_n)
    on_aug = top_up(on_real, X_syn_on, target_n)

    X_aug = np.vstack([X_train[y_train == HC], off_aug, on_aug])
    y_aug = np.concatenate([
        np.full(n_hc, HC, dtype=int),
        np.full(len(off_aug), PD_OFF, dtype=int),
        np.full(len(on_aug), PD_ON, dtype=int),
    ])
    shuffle_idx = rng.permutation(len(X_aug))
    return X_aug[shuffle_idx], y_aug[shuffle_idx]


# --------------------------------------------------
# 3. CLASSIFIERS
# --------------------------------------------------

def get_classifiers():
    return {
        "Logistic Regression": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=7),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
        "Gaussian Naive Bayes": GaussianNB(),
    }


def evaluate_3class(name, clf, X_train, y_train, X_test, y_test):
    sc = StandardScaler()
    Xtr, Xte = sc.fit_transform(X_train), sc.transform(X_test)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)

    return {
        "Classifier": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro F1": f1_score(y_test, pred, average="macro", zero_division=0),
        "F1 (HC)": f1_score(y_test, pred, labels=[HC], average="macro", zero_division=0),
        "F1 (PD-OFF)": f1_score(y_test, pred, labels=[PD_OFF], average="macro", zero_division=0),
        "F1 (PD-ON)": f1_score(y_test, pred, labels=[PD_ON], average="macro", zero_division=0),
    }


def evaluate_binary(name, clf, X_train, y_train, X_test, y_test):
    """y in {0, 1} for a binary task (used for the medication-state classifier)."""
    sc = StandardScaler()
    Xtr, Xte = sc.fit_transform(X_train), sc.transform(X_test)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    prob = clf.predict_proba(Xte)[:, 1] if hasattr(clf, "predict_proba") else clf.decision_function(Xte)
    try:
        auc = roc_auc_score(y_test, prob)
    except ValueError:
        auc = 0.5

    return {
        "Classifier": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": auc,
    }


def run_3class_experiment(X_train, y_train, X_test, y_test, label):
    print(f"\n{'='*60}\nEXPERIMENT: {label}\n{'='*60}")
    print(f"Train: {X_train.shape}  HC={np.sum(y_train==HC)}  OFF={np.sum(y_train==PD_OFF)}  ON={np.sum(y_train==PD_ON)}")
    results = []
    for name, clf in get_classifiers().items():
        r = evaluate_3class(name, clf, X_train, y_train, X_test, y_test)
        results.append(r)
        print(f"  {name:<22} Acc={r['Accuracy']:.3f}  MacroF1={r['Macro F1']:.3f}")
    return pd.DataFrame(results)


def run_binary_experiment(X_train, y_train, X_test, y_test, label):
    print(f"\n{'='*60}\nEXPERIMENT: {label}\n{'='*60}")
    print(f"Train: {X_train.shape}  OFF={np.sum(y_train==0)}  ON={np.sum(y_train==1)}")
    results = []
    for name, clf in get_classifiers().items():
        r = evaluate_binary(name, clf, X_train, y_train, X_test, y_test)
        results.append(r)
        print(f"  {name:<22} Acc={r['Accuracy']:.3f}  F1={r['F1']:.3f}  AUC={r['ROC-AUC']:.3f}")
    return pd.DataFrame(results)


# --------------------------------------------------
# MAIN EXECUTION
# --------------------------------------------------

if __name__ == "__main__":

    X, y, patient_ids = load_medication_labeled_epochs()
    synthetics_off, synthetics_on = load_synthetic_medication()

    (X_train, y_train, train_pids,
     X_test, y_test, test_pids) = patient_level_split_3class(X, y, patient_ids)
    print(f"\nTrain: {X_train.shape}  Test: {X_test.shape}")

    # ============================================================
    # PART A -- 3-CLASS DIAGNOSTIC CLASSIFICATION (HC vs PD-OFF vs PD-ON)
    # ============================================================
    all_3class = {}
    X_train_bal, y_train_bal = balance_3class(X_train, y_train)
    all_3class["Original only"] = run_3class_experiment(X_train_bal, y_train_bal, X_test, y_test, "Original only")

    for src in synthetics_off:
        X_syn_off = synthetics_off.get(src)
        X_syn_on = synthetics_on.get(src)
        X_aug, y_aug = augment_with_synthetic_3class(X_train, y_train, X_syn_off, X_syn_on)
        label = f"+ {src} Synthetic (Balanced)"
        all_3class[label] = run_3class_experiment(X_aug, y_aug, X_test, y_test, label)

    # Comparison plot
    if len(all_3class) > 1:
        clfs = list(all_3class["Original only"]["Classifier"])
        x = np.arange(len(clfs))
        n_exp = len(all_3class)
        width = 0.8 / n_exp
        plt.figure(figsize=(13, 5))
        palette = ["tomato", "steelblue", "gold", "mediumseagreen", "mediumpurple"]
        for idx, (exp_label, df) in enumerate(all_3class.items()):
            offset = x - (width * n_exp / 2) + (width * idx) + (width / 2)
            plt.bar(offset, df["Macro F1"].values, width, label=exp_label, color=palette[idx % len(palette)], alpha=0.85)
        plt.xticks(x, clfs, rotation=20, ha="right", fontsize=9)
        plt.ylabel("Macro F1")
        plt.title("3-Class (HC / PD-OFF / PD-ON) Macro F1 — Real Test Set")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    combined_3class = []
    for label, df in all_3class.items():
        d = df.copy()
        d["Experiment"] = label
        combined_3class.append(d)
    pd.concat(combined_3class, ignore_index=True).to_csv("classification_results_medication_3class.csv", index=False)
    print("\nSaved -> classification_results_medication_3class.csv")

    # ============================================================
    # PART B -- MEDICATION-STATE CLASSIFIER (PD-OFF vs PD-ON, HC excluded)
    # ============================================================
    # "Prove that synthetic data can train a model to detect whether a
    # patient has taken their medication."
    off_on_mask_train = np.isin(y_train, [PD_OFF, PD_ON])
    off_on_mask_test = np.isin(y_test, [PD_OFF, PD_ON])

    X_train_state = X_train[off_on_mask_train]
    y_train_state = (y_train[off_on_mask_train] == PD_ON).astype(int)  # 0=OFF, 1=ON
    X_test_state = X_test[off_on_mask_test]
    y_test_state = (y_test[off_on_mask_test] == PD_ON).astype(int)

    all_state = {}
    all_state["Real only"] = run_binary_experiment(
        X_train_state, y_train_state, X_test_state, y_test_state, "Medication state: Real only"
    )

    for src in synthetics_off:
        X_syn_off = synthetics_off.get(src)
        X_syn_on = synthetics_on.get(src)
        if X_syn_off is None or X_syn_on is None:
            continue
        X_aug_state = np.vstack([X_train_state, X_syn_off, X_syn_on])
        y_aug_state = np.concatenate([
            y_train_state,
            np.zeros(len(X_syn_off), dtype=int),
            np.ones(len(X_syn_on), dtype=int),
        ])
        label = f"Medication state: Real + Synth {src}"
        all_state[label] = run_binary_experiment(X_aug_state, y_aug_state, X_test_state, y_test_state, label)

    combined_state = []
    for label, df in all_state.items():
        d = df.copy()
        d["Experiment"] = label
        combined_state.append(d)
    pd.concat(combined_state, ignore_index=True).to_csv("classification_results_medication_state.csv", index=False)
    print("\nSaved -> classification_results_medication_state.csv")

    # ============================================================
    # PART C -- MEDICATION-STATE CLASSIFIER IN THE CAE'S OWN LATENT SPACE
    # ============================================================
    # Does classifying directly in the 22-D space the CAE's contrastive loss
    # built for exactly this separation do better than Part B's "decode back
    # to 315-D raw features, then re-learn the separation from scratch"?
    cae_encoder = load_cae_encoder()
    if cae_encoder is not None and os.path.exists("cae_scaler_med.pkl"):
        with open("cae_scaler_med.pkl", "rb") as f:
            cae_scaler = pickle.load(f)

        Z_train_state = encode_with_cae(X_train_state, cae_encoder, cae_scaler)
        Z_test_state = encode_with_cae(X_test_state, cae_encoder, cae_scaler)

        all_state_latent = {}
        all_state_latent["Real only"] = run_binary_experiment(
            Z_train_state, y_train_state, Z_test_state, y_test_state,
            "Medication state (CAE latent space): Real only",
        )

        # Synthetic CAE latents are used in their NATIVE 22-D form here -- no
        # decode-to-315D step, which is the entire point of this experiment.
        syn_off_file = "synthetic_pd_off_cae_med.npz"
        syn_on_file = "synthetic_pd_on_cae_med.npz"
        if os.path.exists(syn_off_file) and os.path.exists(syn_on_file):
            Z_syn_off = np.load(syn_off_file)["Z"]
            Z_syn_on = np.load(syn_on_file)["Z"]

            Z_aug_state = np.vstack([Z_train_state, Z_syn_off, Z_syn_on])
            y_aug_state_latent = np.concatenate([
                y_train_state,
                np.zeros(len(Z_syn_off), dtype=int),
                np.ones(len(Z_syn_on), dtype=int),
            ])
            all_state_latent["Real + Synth CAE"] = run_binary_experiment(
                Z_aug_state, y_aug_state_latent, Z_test_state, y_test_state,
                "Medication state (CAE latent space): Real + Synth CAE",
            )
        else:
            print("  [INFO] Synthetic CAE latent files not found -- skipping the augmented latent-space experiment.")

        combined_state_latent = []
        for label, df in all_state_latent.items():
            d = df.copy()
            d["Experiment"] = label
            combined_state_latent.append(d)
        pd.concat(combined_state_latent, ignore_index=True).to_csv(
            "classification_results_medication_state_cae_latent.csv", index=False
        )
        print("\nSaved -> classification_results_medication_state_cae_latent.csv")

        print("\n" + "=" * 70)
        print("RAW 315-D FEATURES  vs  CAE 22-D LATENT SPACE -- same test patients")
        print("=" * 70)
        for label in all_state:
            raw_df = all_state[label]
            raw_best = raw_df.loc[raw_df["F1"].idxmax()]
            print(f"  [raw 315-D]    {label:<45} best={raw_best['Classifier']:<22} "
                  f"F1={raw_best['F1']:.3f}  AUC={raw_best['ROC-AUC']:.3f}")
        for label in all_state_latent:
            latent_df = all_state_latent[label]
            latent_best = latent_df.loc[latent_df["F1"].idxmax()]
            print(f"  [CAE 22-D]     {label:<45} best={latent_best['Classifier']:<22} "
                  f"F1={latent_best['F1']:.3f}  AUC={latent_best['ROC-AUC']:.3f}")
    else:
        print("\n[INFO] Skipping Part C -- contrastive_autoencoder_weights_med.pth or "
              "cae_scaler_med.pkl not found (need the full CAE model, not just its decoder).")


## Downstream Generalization — Train-on-Synthetic, Test-on-Real (TSTR)

Exp 1 / 1b: Real-HC + Synth-PD-OFF (or -ON) -> unseen Real-HC + Real-PD-OFF/ON (diagnostic sensitivity). Exp 2: Synth-PD-OFF vs Synth-PD-ON **only** (no real PD in training at all) -> unseen Real-PD-OFF vs Real-PD-ON (medication-state monitoring, the literal train-synthetic-test-real claim). All patient-grouped.

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

HC, PD_OFF, PD_ON = 0, 1, 2
RANDOM_STATE = 42
# NOTE: the pool/holdout fraction is now fixed by HOLDOUT_FRACTION in the
# labels cell (shared by every encoder/DDPM/classifier in this notebook), not
# by a local TEST_SIZE here -- see patient_level_split_3class() below.

# --------------------------------------------------
# WHAT THIS CELL RUNS (per the requested TSTR script)
# --------------------------------------------------
# Exp 1  (diagnostic sensitivity, OFF): Train = Real-HC-train + Synth-PD-OFF
#         Test  = unseen Real-HC + unseen Real-PD-OFF patients
# Exp 1b (diagnostic sensitivity, ON):  same, for the ON state
# Exp 2  (medication-state monitoring, true TSTR): Train = Synth-PD-OFF +
#         Synth-PD-ON ONLY (no real PD at all) ; Test = unseen Real-PD-OFF vs
#         unseen Real-PD-ON patients -- proves whether synthetic data alone
#         can train a model that detects medication state in real patients.
#
# All three experiments use a PATIENT-GROUPED split: a patient's epochs never
# appear in both train and test.


# --------------------------------------------------
# DECODER LOADING (duplicated for a self-contained cell, per this notebook's
# convention -- see the evaluation / ML cells for the same pattern)
# --------------------------------------------------

class GenericDecoder(nn.Module):
    def __init__(self, latent_dim=22, input_dim=315):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(256, input_dim),
        )

    def forward(self, z):
        return self.decoder(z)


def load_decoder_safely(decoder_path, latent_dim=22, output_dim=315):
    loaded_data = torch.load(decoder_path, map_location="cpu")
    if isinstance(loaded_data, nn.Module):
        model = loaded_data
    elif isinstance(loaded_data, dict):
        model = GenericDecoder(latent_dim=latent_dim, input_dim=output_dim)
        keys = list(loaded_data.keys())
        if any(k.startswith("decoder.") or k.startswith("encoder.") for k in keys):
            model.load_state_dict(loaded_data, strict=False)
        else:
            model.decoder.load_state_dict(loaded_data)
    else:
        raise ValueError(f"Unknown data type inside {decoder_path}: {type(loaded_data)}")
    model.eval()
    return model


SOURCE_CONFIGS = {
    "AE": (["autoencoder_decoder_weights_med.pth"], "latent_autoencoder_med.npz", True, "ae_scaler_med.pkl"),
    "CAE": (["contrastive_decoder_weights_med.pth", "contrastive_autoencoder_decoder_med.pth"],
             "latent_contrastive_med.npz", False, "cae_scaler_med.pkl"),
    "VAE": (["vae_decoder_weights_med.pth", "vae_decoder_med.pth"], "latent_vae_med.npz", True, "vae_scaler_med.pkl"),
}


def decode_medication_synthetic(name, state_tag):
    decoder_files, meta_file, needs_znorm, feature_scaler_file = SOURCE_CONFIGS[name]
    syn_file = f"synthetic_pd_{state_tag}_{name.lower()}_med.npz"
    decoder_file = next((f for f in decoder_files if os.path.exists(f)), None)
    if not (os.path.exists(syn_file) and decoder_file):
        return None

    data = np.load(syn_file)
    Z_syn = data["Z_synthetic"] if "Z_synthetic" in data else data["Z"]

    if needs_znorm and os.path.exists(meta_file):
        meta = np.load(meta_file)
        if "z_mean" in meta and "z_std" in meta and meta["z_mean"].shape[0] == Z_syn.shape[1]:
            Z_syn = (Z_syn * meta["z_std"]) + meta["z_mean"]

    decoder = load_decoder_safely(decoder_file)
    with torch.no_grad():
        X_scaled = decoder(torch.FloatTensor(Z_syn)).numpy()

    X_raw = X_scaled
    if os.path.exists(feature_scaler_file):
        with open(feature_scaler_file, "rb") as f:
            feature_scaler = pickle.load(f)
        scale_ref = getattr(feature_scaler, "scale_", getattr(feature_scaler, "mean_", None))
        if scale_ref is not None and scale_ref.shape[0] == X_scaled.shape[1]:
            X_raw = feature_scaler.inverse_transform(X_scaled)

    return np.clip(X_raw, a_min=1e-6, a_max=None)


def load_synthetic_medication():
    synthetics_off, synthetics_on = {}, {}
    for name in SOURCE_CONFIGS:
        X_off = decode_medication_synthetic(name, "off")
        X_on = decode_medication_synthetic(name, "on")
        if X_off is not None:
            synthetics_off[name] = X_off
        if X_on is not None:
            synthetics_on[name] = X_on
    print(f"Synthetic PD-OFF sources : {sorted(synthetics_off.keys())}")
    print(f"Synthetic PD-ON  sources : {sorted(synthetics_on.keys())}")
    return synthetics_off, synthetics_on


# --------------------------------------------------
# TRAIN/TEST SPLIT -- REUSES THE GLOBAL POOL/HOLDOUT SPLIT
# --------------------------------------------------
# IMPORTANT: must be the SAME split used to decide which patients the
# AE/CAE/VAE encoders and the conditional DDPM were trained on
# (split_pool_and_holdout_patients(), defined in the labels cell). A
# fresh, independently-drawn patient split here -- even a correctly
# patient-grouped one -- would test against patients the GENERATOR had
# already seen, which is exactly the leak that produced the suspiciously
# perfect TSTR scores (AUC=1.000, zero variance across 21 classifier/source
# combinations) found during review. For TSTR in particular, this matters
# more than anywhere else in the notebook: the whole point of Exp2 is "can
# synthetic-only training generalize to patients nothing in the pipeline has
# seen" -- which this split is what actually makes true.

def patient_level_split_3class(X, y, patient_ids, random_state=SPLIT_RANDOM_STATE):
    """Thin wrapper around the canonical pool/holdout split: pool -> "train",
    holdout -> "test". Kept under this name so the rest of this cell doesn't
    need to change."""
    pool_patients, holdout_patients = split_pool_and_holdout_patients(
        y, patient_ids, random_state=random_state
    )
    train_mask, test_mask = get_pool_and_holdout_masks(patient_ids, pool_patients, holdout_patients)
    assert not (set(patient_ids[train_mask]) & set(patient_ids[test_mask])), "PATIENT LEAKAGE DETECTED"

    return train_mask, test_mask


# --------------------------------------------------
# CLASSIFIERS & METRICS
# --------------------------------------------------

def get_classifiers():
    return {
        "Logistic Regression": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=7),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
        "Gaussian Naive Bayes": GaussianNB(),
    }


def patient_level_metrics(probabilities, y_test, patient_ids_test):
    patient_pred, patient_prob, patient_label = [], [], []
    for pid in np.unique(patient_ids_test):
        mask = patient_ids_test == pid
        mean_prob = np.mean(probabilities[mask])
        patient_pred.append(int(mean_prob >= 0.5))
        patient_prob.append(mean_prob)
        patient_label.append(y_test[mask][0])

    try:
        patient_auc = roc_auc_score(patient_label, patient_prob)
    except ValueError:
        patient_auc = 0.5

    return {
        "Patient Accuracy": accuracy_score(patient_label, patient_pred),
        "Patient F1": f1_score(patient_label, patient_pred, zero_division=0),
        "Patient ROC-AUC": patient_auc,
        "N Patients": len(patient_label),
    }


def evaluate_binary(name, clf, X_train, y_train, X_test, y_test, patient_ids_test):
    sc = StandardScaler()
    Xtr, Xte = sc.fit_transform(X_train), sc.transform(X_test)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    prob = clf.predict_proba(Xte)[:, 1] if hasattr(clf, "predict_proba") else clf.decision_function(Xte)
    try:
        auc = roc_auc_score(y_test, prob)
    except ValueError:
        auc = 0.5

    result = {
        "Classifier": name,
        "Epoch Accuracy": accuracy_score(y_test, pred),
        "Epoch Precision": precision_score(y_test, pred, zero_division=0),
        "Epoch Recall": recall_score(y_test, pred, zero_division=0),
        "Epoch F1": f1_score(y_test, pred, zero_division=0),
        "Epoch ROC-AUC": auc,
    }
    result.update(patient_level_metrics(prob, y_test, patient_ids_test))
    return result


def run_experiment(X_train, y_train, X_test, y_test, patient_ids_test, label):
    print(f"\n{'='*70}\nEXPERIMENT: {label}\n{'='*70}")
    print(f"Train: {X_train.shape}  pos(1)={np.sum(y_train==1)}  neg(0)={np.sum(y_train==0)}")
    print(f"Test : {X_test.shape}  ({len(np.unique(patient_ids_test))} patients)")
    results = []
    for name, clf in get_classifiers().items():
        r = evaluate_binary(name, clf, X_train, y_train, X_test, y_test, patient_ids_test)
        results.append(r)
        print(f"  {name:<22} Epoch F1={r['Epoch F1']:.3f}  Patient F1={r['Patient F1']:.3f}  Patient AUC={r['Patient ROC-AUC']:.3f}")
    return pd.DataFrame(results)


def balance_binary(X, y, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = min(np.sum(y == 0), np.sum(y == 1))
    idx0 = rng.choice(np.where(y == 0)[0], size=n, replace=False)
    idx1 = rng.choice(np.where(y == 1)[0], size=n, replace=False)
    sel = rng.permutation(np.concatenate([idx0, idx1]))
    return X[sel], y[sel]


# --------------------------------------------------
# MAIN EXECUTION
# --------------------------------------------------

if __name__ == "__main__":

    X, y, patient_ids = load_medication_labeled_epochs()
    synthetics_off, synthetics_on = load_synthetic_medication()

    train_mask, test_mask = patient_level_split_3class(X, y, patient_ids)
    X_train, y_train, pid_train = X[train_mask], y[train_mask], patient_ids[train_mask]
    X_test, y_test, pid_test = X[test_mask], y[test_mask], patient_ids[test_mask]

    real_hc_train = X_train[y_train == HC]
    real_off_train = X_train[y_train == PD_OFF]
    real_on_train = X_train[y_train == PD_ON]

    all_results = {}

    # ============================================================
    # EXPERIMENT 1 -- Diagnostic sensitivity, PD-OFF
    # Train: Real-HC-train + Synth-PD-OFF   Test: unseen Real-HC + Real-PD-OFF
    # ============================================================
    hc_mask_test = np.isin(y_test, [HC, PD_OFF])
    X_test_off_task = X_test[hc_mask_test]
    y_test_off_task = (y_test[hc_mask_test] == PD_OFF).astype(int)
    pid_test_off_task = pid_test[hc_mask_test]

    for src, X_syn_off in synthetics_off.items():
        X_tr = np.vstack([real_hc_train, X_syn_off])
        y_tr = np.concatenate([np.zeros(len(real_hc_train), dtype=int), np.ones(len(X_syn_off), dtype=int)])
        X_tr, y_tr = balance_binary(X_tr, y_tr)
        label = f"Exp1 [PD-OFF sensitivity] Real-HC + Synth-{src}-OFF"
        all_results[label] = run_experiment(X_tr, y_tr, X_test_off_task, y_test_off_task, pid_test_off_task, label)

    # Real-data upper-bound reference (Real HC + Real PD-OFF, both from train)
    X_tr_real, y_tr_real = balance_binary(
        np.vstack([real_hc_train, real_off_train]),
        np.concatenate([np.zeros(len(real_hc_train), dtype=int), np.ones(len(real_off_train), dtype=int)]),
    )
    all_results["Exp1 [PD-OFF sensitivity] Real-HC + Real-OFF (reference)"] = run_experiment(
        X_tr_real, y_tr_real, X_test_off_task, y_test_off_task, pid_test_off_task,
        "Exp1 [PD-OFF sensitivity] Real-HC + Real-OFF (reference)",
    )

    # ============================================================
    # EXPERIMENT 1b -- Diagnostic sensitivity, PD-ON
    # Train: Real-HC-train + Synth-PD-ON   Test: unseen Real-HC + Real-PD-ON
    # ============================================================
    on_mask_test = np.isin(y_test, [HC, PD_ON])
    X_test_on_task = X_test[on_mask_test]
    y_test_on_task = (y_test[on_mask_test] == PD_ON).astype(int)
    pid_test_on_task = pid_test[on_mask_test]

    for src, X_syn_on in synthetics_on.items():
        X_tr = np.vstack([real_hc_train, X_syn_on])
        y_tr = np.concatenate([np.zeros(len(real_hc_train), dtype=int), np.ones(len(X_syn_on), dtype=int)])
        X_tr, y_tr = balance_binary(X_tr, y_tr)
        label = f"Exp1b [PD-ON sensitivity] Real-HC + Synth-{src}-ON"
        all_results[label] = run_experiment(X_tr, y_tr, X_test_on_task, y_test_on_task, pid_test_on_task, label)

    X_tr_real, y_tr_real = balance_binary(
        np.vstack([real_hc_train, real_on_train]),
        np.concatenate([np.zeros(len(real_hc_train), dtype=int), np.ones(len(real_on_train), dtype=int)]),
    )
    all_results["Exp1b [PD-ON sensitivity] Real-HC + Real-ON (reference)"] = run_experiment(
        X_tr_real, y_tr_real, X_test_on_task, y_test_on_task, pid_test_on_task,
        "Exp1b [PD-ON sensitivity] Real-HC + Real-ON (reference)",
    )

    # ============================================================
    # EXPERIMENT 2 -- Medication-state monitoring (TRUE TSTR)
    # Train: Synth-PD-OFF + Synth-PD-ON ONLY (no real PD at all)
    # Test:  unseen Real-PD-OFF vs unseen Real-PD-ON
    # ============================================================
    state_mask_test = np.isin(y_test, [PD_OFF, PD_ON])
    X_test_state = X_test[state_mask_test]
    y_test_state = (y_test[state_mask_test] == PD_ON).astype(int)
    pid_test_state = pid_test[state_mask_test]

    for src in synthetics_off:
        if src not in synthetics_on:
            continue
        X_syn_off, X_syn_on = synthetics_off[src], synthetics_on[src]
        X_tr = np.vstack([X_syn_off, X_syn_on])
        y_tr = np.concatenate([np.zeros(len(X_syn_off), dtype=int), np.ones(len(X_syn_on), dtype=int)])
        X_tr, y_tr = balance_binary(X_tr, y_tr)
        label = f"Exp2 [TSTR med-state] Synth-{src}-OFF vs Synth-{src}-ON only"
        all_results[label] = run_experiment(X_tr, y_tr, X_test_state, y_test_state, pid_test_state, label)

    # Real-data upper-bound reference (train on real OFF/ON directly)
    X_tr_real, y_tr_real = balance_binary(
        np.vstack([real_off_train, real_on_train]),
        np.concatenate([np.zeros(len(real_off_train), dtype=int), np.ones(len(real_on_train), dtype=int)]),
    )
    all_results["Exp2 [TSTR med-state] Real-OFF vs Real-ON (reference)"] = run_experiment(
        X_tr_real, y_tr_real, X_test_state, y_test_state, pid_test_state,
        "Exp2 [TSTR med-state] Real-OFF vs Real-ON (reference)",
    )

    # ============================================================
    # SAVE
    # ============================================================
    combined = []
    for label, df in all_results.items():
        d = df.copy()
        d["Experiment"] = label
        combined.append(d)
    final_df = pd.concat(combined, ignore_index=True)
    final_df.to_csv("downstream_generalization_results_medication.csv", index=False)
    print("\nSaved -> downstream_generalization_results_medication.csv")

    print("\n" + "=" * 70)
    print("SUMMARY (Patient-level F1, mean across classifiers)")
    print("=" * 70)
    summary = final_df.groupby("Experiment")["Patient F1"].mean().sort_values(ascending=False)
    print(summary.to_string())
